# Pricing one circuit: carbon on the 4^3 lattice

This notebook works one of the lecture's calculations, the cost of one circuit of
carbon's neutrino response. It is the companion of hour 2: its sections follow hour 2's
slides from "Welcome back", under the slides' own titles (one is shortened), and fill in
what the slides leave out, the math behind each step and the code behind each number.
Read the two side by side.

Two kinds of code cell do the work. Most call the tested module `response_circuit`, at
`src/qec_utility_estimation/apps/pionless_first_order/response_circuit.py`, and print
its counts, converting units where that reads better: a count as a share of a budget,
seconds as days. The others build small circuits with Cirq, gate by gate, so that you
can see what the slides draw. Each of those is checked by an `assert` in a cell after
it, against something independent of it, such as Cirq's own gates or the energies of
the Hamiltonian, so a circuit built wrong stops the notebook with an error.

The outputs are saved, so the notebook reads on the repository's web page with nothing
installed. To run it yourself, from the repository's top level:

```
uv sync --locked
uv run jupyter lab
```

then open `notebooks/price_one_circuit.ipynb` and run its cells from the top. The first
time Cirq is imported on a computer it may print "Matplotlib is building the font cache;
this may take a moment." That happens once. The script `scripts/price_one_circuit.py`
prints the module's side of the same calculation, and its options run helium, argon and
the exercises.

## Welcome back

Hour 1 set up the problem. DUNE needs the neutrino response of nuclei, and for carbon
that is already a supercomputer campaign. A quantum computer holds a nucleus's state
natively and runs real time natively, and its output is always samples. On a
fault-tolerant machine the Clifford gates are nearly free and every $T$ gate costs a
magic state, so we count $T$ gates, logical qubits and the error per gate. The 2033
budget is $10^9$ hard operations per circuit on about $10^3$ logical qubits.

Hour 2 prices one calculation against that budget, piece by piece: the inclusive
response of carbon on the $4^3$ lattice. The next cells set up the case; the circuit
words follow.

## The case

`LECTURE_CASES["carbon"]` is the lecture's case: Carbon-12 on the $4^3$ lattice, with
its energy register sized at 10 MeV.

The price depends on one number known only roughly in advance: the *strength*
$\eta^2 = \|J|\Psi_0\rangle\|^2$. As hour 1's slide "The object: the response is a
histogram of final-state energies" puts it, applying the current $J$ to the ground state
gives a superposition of final states, and their weights make the histogram. The
strength is the histogram's total weight at one momentum transfer $\vec q$, summed over
all energies. The circuit applies $J$ with a step that rarely succeeds: its probability
is proportional to $\eta^2$. So the circuit repeats the ground state's preparation and
that step many times to make success near certain, and a weaker response costs more. The
slides "Applying the current: a block encoding that rarely succeeds" and "Amplitude
amplification: making a rare success near certain" show how.

We take the strength to lie between half and all of carbon's proton count $Z$, so
$Z/2 \le \eta^2 \le Z$. That range is the *strength band*. It is our estimate, and the
docstring of `Nucleus.strength_band` gives the reasons for it. `price_lecture_case`
prices one circuit at each end of the band and returns them weak end first: `weak` at
$\eta^2 = Z/2$, the costlier, and `strong` at $\eta^2 = Z$. Most cells quote the weak
end. Where both ends matter, a cell prints the range from low to high, as the slides do.

In [1]:
import itertools
import math

import cirq
import numpy as np

from qec_utility_estimation.apps.pionless_first_order.norms import pauli_coefficients
from qec_utility_estimation.apps.pionless_first_order.response_circuit import (
    FAULT_BUDGET,
    HARD_OPERATIONS_2028,
    HARD_OPERATIONS_2033,
    LATTICE_SPACING_FM,
    LECTURE_CASES,
    LECTURE_COUPLINGS,
    LOGICAL_ERROR_2028,
    MODES_PER_SITE,
    SYNTHESIS_ERROR_BUDGET,
    T_PER_TOFFOLI,
    TERMS_PER_SITE,
    LectureCase,
    amplification,
    amplitudes_after_translations,
    bins_in_window,
    enumeration_register_qubits,
    expected_faults,
    gain_at_proton_count,
    ground_state_amplitudes,
    ground_state_load,
    logical_error_rate,
    one_site_one_norm,
    physical_qubits,
    physical_qubits_per_logical,
    pinned_table_amplitudes,
    position_register_qubits,
    price_lecture_case,
    price_strength_band,
    randomized_synthesis_precision,
    readout_rotations,
    register_bits_for_sizing,
    ross_selinger_t_estimate,
    runs_for_the_set,
    runs_to_pin_bin_to_itself,
    runs_to_pin_bin_to_strength,
    standing,
    standing_if_one_per_toffoli,
    synthesis_budget,
    wall_time_seconds,
)
from qec_utility_estimation.qubitization import (
    equal_superposition_cost,
    unary_iteration_select_toffoli_count,
)
from qec_utility_estimation.synthesis import repeat_until_success_t_count

In [2]:
carbon = LECTURE_CASES["carbon"]
weak, strong = price_lecture_case(carbon)
weakest, strongest = carbon.nucleus.strength_band
print(carbon)
print(
    f"One circuit for {weak.nucleus.name} on the {weak.lattice.box_side}^3 "
    f"lattice, with {weak.register.bits} energy bits"
)
print(
    f"strength band: {weakest} to {strongest} "
    f"(Z = {carbon.nucleus.proton_count} protons)"
)

LectureCase(nucleus=Nucleus(name='carbon', nucleons_per_species=(3, 3, 3, 3)), box_side=4, sizing_mev=10.0)
One circuit for carbon on the 4^3 lattice, with 12 energy bits
strength band: 3.0 to 6.0 (Z = 6 protons)


# Carbon on qubits

The first part of hour 2 gives the circuit words it needs, then writes carbon and its
Hamiltonian on qubits.

## Circuit words for hour 2 (i): controlled gates, Toffoli, AND

A controlled gate applies a gate $U$ to its target only when the control qubit is
$|1\rangle$. The CNOT is the controlled $X$: $|a, b\rangle \to |a, b \oplus a\rangle$.
In Cirq's drawings time runs left to right, a control is `@`, the target of a CNOT is
`X`, and a gate's inverse carries the exponent $-1$, so $T^\dagger$ is `T^-1`.

In [3]:
a, b, c = cirq.NamedQubit("a"), cirq.NamedQubit("b"), cirq.NamedQubit("c")
cnot = cirq.Circuit(cirq.CNOT(a, b))
print(cnot.to_text_diagram(qubit_order=[a, b]))
for bits in itertools.product([0, 1], repeat=2):
    final = cirq.final_state_vector(
        cnot, initial_state=list(bits), qubit_order=[a, b], dtype=np.complex128
    )
    print(f"|{bits[0]}{bits[1]}> -> |{int(np.argmax(np.abs(final))):02b}>")

a: ───@───
      │
b: ───X───
|00> -> |00>
|01> -> |01>
|10> -> |11>
|11> -> |10>


The Toffoli is the controlled-controlled-NOT,
$|a, b, c\rangle \to |a, b, c \oplus (a \wedge b)\rangle$. It is not a Clifford gate. The
textbook circuit for it (Nielsen and Chuang, Fig. 4.9) needs seven $T$ or $T^\dagger$
gates, with $T = \mathrm{diag}(1, e^{i\pi/4})$, among Hadamards and CNOTs. The next cell
builds a circuit of that kind gate by gate; the cell after checks that its matrix is
Cirq's own Toffoli, and counts its $T$ gates.

In [4]:
T_DAGGER = cirq.T**-1
toffoli = cirq.Circuit(
    cirq.H(c),
    cirq.CNOT(b, c),
    T_DAGGER(c),
    cirq.CNOT(a, c),
    cirq.T(c),
    cirq.CNOT(b, c),
    T_DAGGER(c),
    cirq.CNOT(a, c),
    cirq.T(b),
    cirq.T(c),
    cirq.H(c),
    cirq.CNOT(a, b),
    cirq.T(a),
    T_DAGGER(b),
    cirq.CNOT(a, b),
)
print(toffoli.to_text_diagram(qubit_order=[a, b, c]))

a: ──────────────────@──────────────────@───@───T──────@───
                     │                  │   │          │
b: ───────@──────────┼───────@───T──────┼───X───T^-1───X───
          │          │       │          │
c: ───H───X───T^-1───X───T───X───T^-1───X───T───H──────────


In [5]:
toffoli_t = 0
for operation in toffoli.all_operations():
    if operation.gate in (cirq.T, T_DAGGER):
        toffoli_t += 1
assert np.allclose(toffoli.unitary(qubit_order=[a, b, c]), cirq.unitary(cirq.TOFFOLI))
assert toffoli_t == 7
print("the circuit's matrix is the Toffoli's")
print(f"its T and T^-1 gates: {toffoli_t}")

the circuit's matrix is the Toffoli's
its T and T^-1 gates: 7


An AND is a Toffoli whose target starts in $|0\rangle$: it writes $a \wedge b$ into a
fresh qubit, $|a, b, 0\rangle \to |a, b, a \wedge b\rangle$. Knowing that the target
starts in $|0\rangle$ lets Gidney (Quantum 2, 74 (2018)) do it with four $T$ or
$T^\dagger$ gates. The target is prepared in $T|+\rangle$. CNOTs then spread parities of
$a$, $b$ and the target over the three qubits, so that three more $T$ or $T^\dagger$
gates give the branch $x$ of the target the phase $(-1)^{x(a \wedge b)}$, times
$(-i)^{a \wedge b}$. A Hadamard turns that phase into the bit $a \wedge b$, and an $S$
removes the $(-i)^{a \wedge b}$. The next cell builds it; the cell after checks it on its
four inputs, phases included, and counts its $T$ gates.

In [6]:
and_gate = cirq.Circuit(
    cirq.H(c),
    cirq.T(c),
    cirq.CNOT(a, c),
    cirq.CNOT(b, c),
    cirq.CNOT(c, a),
    cirq.CNOT(c, b),
    T_DAGGER(a),
    T_DAGGER(b),
    cirq.T(c),
    cirq.CNOT(c, a),
    cirq.CNOT(c, b),
    cirq.H(c),
    cirq.S(c),
)
print(and_gate.to_text_diagram(qubit_order=[a, b, c]))

a: ───────────@───────X───T^-1──────────X───────────────
              │       │                 │
b: ───────────┼───@───┼───X──────T^-1───┼───X───────────
              │   │   │   │             │   │
c: ───H───T───X───X───@───@──────T──────@───@───H───S───


In [7]:
and_t = 0
for operation in and_gate.all_operations():
    if operation.gate in (cirq.T, T_DAGGER):
        and_t += 1
for bits in itertools.product([0, 1], repeat=2):
    final = cirq.final_state_vector(
        and_gate, initial_state=[*bits, 0], qubit_order=[a, b, c], dtype=np.complex128
    )
    both = bits[0] * bits[1]
    expected = np.zeros(8)
    expected[4 * bits[0] + 2 * bits[1] + both] = 1
    assert np.allclose(final, expected)
    print(f"|{bits[0]}{bits[1]}0> -> |{bits[0]}{bits[1]}{both}>")
assert and_t == 4
print(f"its T and T^-1 gates: {and_t}")

|000> -> |000>
|010> -> |010>
|100> -> |100>
|110> -> |111>
its T and T^-1 gates: 4


Erasing an AND costs no $T$ gate. Measure the target in the $X$ basis, a Hadamard and
then a measurement. With the target holding $a \wedge b$, the Hadamard gives
$\sum_{a,b} \alpha_{ab}\, |a, b\rangle \left(|0\rangle + (-1)^{a \wedge b}|1\rangle\right)/\sqrt{2}$.
The outcome is random. On 0 the system is back where it started; on 1 it carries the
sign $(-1)^{a \wedge b}$, which a CZ on $a$ and $b$ removes, a Clifford gate run only
when the outcome says so. The next cell draws the erasure; the cell after starts from a
random state of $a$ and $b$, computes the AND, erases it, and checks that $a$ and $b$ end
where they started, whichever the outcome.

In [8]:
erase = cirq.Circuit(
    cirq.H(c),
    cirq.measure(c, key="outcome"),
    cirq.CZ(a, b).with_classical_controls("outcome"),
)
print(erase.to_text_diagram(qubit_order=[a, b, c]))

a: ─────────────────@───
                    ║
b: ─────────────────@───
                    ║
c: ─────────H───M───╫───
                ║   ║
outcome: ═══════@═══^═══


In [9]:
generator = np.random.default_rng(seed=12)
start = generator.normal(size=4) + 1j * generator.normal(size=4)
start /= np.linalg.norm(start)
computed = cirq.final_state_vector(
    and_gate,
    initial_state=np.kron(start, [1, 0]),
    qubit_order=[a, b, c],
    dtype=np.complex128,
)
outcomes = set()
for seed in range(8):
    result = cirq.Simulator(dtype=np.complex128, seed=seed).simulate(
        erase, initial_state=computed, qubit_order=[a, b, c]
    )
    outcome = int(result.measurements["outcome"][0])
    outcomes.add(outcome)
    assert np.allclose(result.final_state_vector, np.kron(start, np.eye(2)[outcome]))
assert outcomes == {0, 1}
print(f"a and b end where they started, on outcomes {sorted(outcomes)}")

a and b end where they started, on outcomes [0, 1]


The lecture prices every Toffoli at 4 $T$, as Babbush et al. do (arXiv:1805.03662): our
Toffolis are ANDs that are erased afterward, so each costs Gidney's four $T$ and its
erasure nothing. The module holds that price in `T_PER_TOFFOLI`. The budget's wording
could also let a Toffoli count as one hard operation if you had a native Toffoli. The
lecture keeps 4, the safe side for a claim that a circuit fits, and gives the other
count where it changes a claim.

In [10]:
assert T_PER_TOFFOLI == and_t
print(f"the module's price of a Toffoli: {T_PER_TOFFOLI} T, the AND's T gates")

the module's price of a Toffoli: 4 T, the AND's T gates


## Circuit words for hour 2 (ii): registers, chains, rotations

A register is a named group of qubits that holds one quantity. Hour 2 has a system
register (the nucleus, one qubit per lattice mode), an energy register (where phase
estimation writes the energy) and ancilla registers (scratch space). Ancillas start and
end in $|0\rangle$: an ancilla left changed stays entangled with the system and spoils
its state. Running a computation backward to clear its ancillas is uncomputing.

An AND chain tests whether $k$ qubits are all $|1\rangle$. The AND of the first two goes
into a fresh ancilla, then the AND of that ancilla and the third into another, and so
on: $k - 1$ ANDs, $4(k - 1)$ $T$. Hour 2's reflections are AND chains. The next cell
draws the chain for $k = 4$, each AND drawn as a Toffoli onto a fresh ancilla, as the
slides draw it; the cell after checks it on all 16 inputs, reading the last ancilla as
the last bit of the output.

In [11]:
inputs = [cirq.NamedQubit(f"q{i}") for i in range(4)]
chain_ancillas = [cirq.NamedQubit(f"a{i}") for i in range(1, 4)]
and_chain = cirq.Circuit(
    cirq.TOFFOLI(inputs[0], inputs[1], chain_ancillas[0]),
    cirq.TOFFOLI(chain_ancillas[0], inputs[2], chain_ancillas[1]),
    cirq.TOFFOLI(chain_ancillas[1], inputs[3], chain_ancillas[2]),
)
print(and_chain.to_text_diagram(qubit_order=[*inputs, *chain_ancillas]))

q0: ───@───────────
       │
q1: ───@───────────
       │
q2: ───┼───@───────
       │   │
q3: ───┼───┼───@───
       │   │   │
a1: ───X───@───┼───
           │   │
a2: ───────X───@───
               │
a3: ───────────X───


In [12]:
for bits in itertools.product([0, 1], repeat=4):
    final = cirq.final_state_vector(
        and_chain,
        initial_state=[*bits, 0, 0, 0],
        qubit_order=[*inputs, *chain_ancillas],
        dtype=np.complex128,
    )
    last_ancilla = int(np.argmax(np.abs(final))) % 2
    assert last_ancilla == bits[0] * bits[1] * bits[2] * bits[3]
chain_ands = len(chain_ancillas)
print("the last ancilla holds the AND of all four inputs, on all 16 of them")
print(f"{chain_ands} ANDs, {chain_ands * T_PER_TOFFOLI} T")

the last ancilla holds the AND of all four inputs, on all 16 of them
3 ANDs, 12 T


A rotation $R_z(\theta) = \mathrm{diag}(e^{-i\theta/2}, e^{i\theta/2})$ turns the phase
by an arbitrary angle. The $T$ gate is one, up to a global phase:
$T = e^{i\pi/8} R_z(\pi/4)$. With the Clifford gates, $T$ makes the rotations by
multiples of $\pi/4$ exactly. A fault-tolerant machine cannot run any other angle
directly; it builds an approximation from a few dozen $T$ gates, priced in
"Rotations (ii)" below. The currencies, from hour 1: $T$ gates per circuit, logical
qubits, and $\epsilon_l$ per gate, with each Toffoli counted as 4 $T$.

In [13]:
assert np.allclose(
    cirq.unitary(cirq.T), np.exp(1j * math.pi / 8) * cirq.unitary(cirq.rz(math.pi / 4))
)
print("T = exp(i pi / 8) Rz(pi / 4)")

T = exp(i pi / 8) Rz(pi / 4)


## The model: nucleons on a lattice

What do the qubits describe? Nucleons on a periodic cubic lattice, in leading-order
pionless EFT. A nucleon hops to a neighboring site with strength
$t = \hbar^2/(2M_N a^2)$, and nucleons on one site meet through a two-body contact $C_0$
and a three-body contact $D_0$.

`weak.lattice` is the `Lattice` with 4 sites along each axis. The spacing,
`LATTICE_SPACING_FM`, is 1.4 fm, and `LECTURE_COUPLINGS` holds $t$, $C_0$ and $D_0$ at
that spacing, from Roggero et al. 2019 (arXiv:1911.06368), Tab. I, rounded to two
decimals. The contacts were fitted to the deuteron, a two-nucleon scattering length and
the triton on a richer lattice Hamiltonian (Rokash et al., arXiv:1308.3386). On this
simpler one the deuteron binds more deeply than in nature, so read it as a model with
realistic scales.

In [14]:
lattice = weak.lattice
print(f"spacing a: {LATTICE_SPACING_FM} fm")
print(f"sites: {lattice.box_side}^3 = {lattice.site_count}")
print(f"box: {lattice.box_length_fm:.1f} fm on a side")
print(f"hopping t: {LECTURE_COUPLINGS.hopping_scale_mev} MeV")
print(f"two-body contact C0: {LECTURE_COUPLINGS.two_body_mev} MeV")
print(f"three-body contact D0: {LECTURE_COUPLINGS.three_body_mev} MeV")

spacing a: 1.4 fm
sites: 4^3 = 64
box: 5.6 fm on a side
hopping t: 10.58 MeV
two-body contact C0: -98.23 MeV
three-body contact D0: 127.84 MeV


## The Hamiltonian, written out

With $a^\dagger_{i,f}$ creating a nucleon of species $f$ on site $i$, $a_{i,f}$ removing
one, and $n_{i,f} = a^\dagger_{i,f} a_{i,f}$ counting it (0 or 1), the Hamiltonian is
Roggero et al. 2019, Eq. 1:

$$H = 6tA - t \sum_f \sum_{\langle i,j \rangle} \left(a^\dagger_{i,f} a_{j,f} + a^\dagger_{j,f} a_{i,f}\right) + C_0 \sum_i \sum_{f < f'} n_{i,f}\, n_{i,f'} + D_0 \sum_i \sum_{f < f' < f''} n_{i,f}\, n_{i,f'}\, n_{i,f''}.$$

The hopping term moves a nucleon to a neighboring site $\langle i, j \rangle$ and keeps
its species; with the constant $6tA$, for $A$ nucleons, it is the kinetic energy on the
lattice. The contacts only count. A site holding $k$ nucleons, all of different species,
has $\binom{k}{2}$ pairs and $\binom{k}{3}$ triples, so its contact energy is
$C_0\binom{k}{2} + D_0\binom{k}{3}$. No term changes a species, so $H$ conserves each
species' number. Eq. 1 itself writes the constant as $2DtA$, with $D = 3$ dimensions,
and the contacts as $\frac{1}{2}C_0\sum_{f \neq f'}$ and
$\frac{1}{6}D_0\sum_{f \neq f' \neq f''}$ over ordered species, which count each pair
twice and each triple six times; the sums above count each once.

The next cell fills one site up. Without the repulsive $D_0$, four nucleons would sit far
lower: the three-body term stops nuclei from collapsing onto single sites.

In [15]:
C0 = LECTURE_COUPLINGS.two_body_mev
D0 = LECTURE_COUPLINGS.three_body_mev
for nucleons in range(5):
    contact = C0 * math.comb(nucleons, 2) + D0 * math.comb(nucleons, 3)
    print(f"{nucleons} nucleons on one site: {contact:8.2f} MeV")
print(f"4 nucleons without the three-body term, 6 C0: {6 * C0:.2f} MeV")

0 nucleons on one site:     0.00 MeV
1 nucleons on one site:     0.00 MeV
2 nucleons on one site:   -98.23 MeV
3 nucleons on one site:  -166.85 MeV
4 nucleons on one site:   -78.02 MeV
4 nucleons without the three-body term, 6 C0: -589.38 MeV


## Two ways to put nucleons on qubits

A site holds four species, $p\!\uparrow$, $p\!\downarrow$, $n\!\uparrow$ and
$n\!\downarrow$. Write $M$ for the number of sites, $M = 4^3 = 64$ on this lattice; $M$
sites hold $4M = 256$ modes, and carbon's $A = 12$ nucleons sit in 12 of them, three in
each species. A many-fermion state can be stored in two ways, two "quantizations".
Occupations (second quantization) give each mode a qubit, $|1\rangle$ occupied and
$|0\rangle$ empty: $4M$ qubits. Fermion signs then live in the operators, and the
Jordan-Wigner strings carry them. Positions (first quantization) give each nucleon a
register naming its mode: $A \lceil \log_2 4M \rceil$ qubits. Fermion signs then live in
the state, which must flip sign when two nucleons' registers are swapped. Both hold the
same states of carbon, $\binom{M}{3}^4$ of them. `Lattice.system_qubits`,
`position_register_qubits` and `ground_state_amplitudes` count them.

In [16]:
nucleus = weak.nucleus
amplitudes = ground_state_amplitudes(nucleus, lattice)
positions = position_register_qubits(nucleus, lattice)
print(f"modes: {lattice.site_count} sites x {MODES_PER_SITE} = {lattice.system_qubits}")
print(f"occupations: {lattice.system_qubits} qubits")
print(
    f"positions: {nucleus.nucleon_count} nucleons x "
    f"{positions // nucleus.nucleon_count} bits = {positions} qubits"
)
print(f"carbon's states, either way: {amplitudes:.2e}")

modes: 64 sites x 4 = 256
occupations: 256 qubits
positions: 12 nucleons x 8 bits = 96 qubits
carbon's states, either way: 3.01e+18


## The register: one qubit per mode

The lecture stores occupations, second quantization. $H$ is written in $a^\dagger$, $a$
and $n$, so each of its terms acts on the qubits of the modes it names. The register is
the list of modes, `MODES_PER_SITE` a site, so it has $4M$ qubits by construction,
`Lattice.system_qubits`. It holds more than carbon: $2^{4M}$ basis states, for any
number of nucleons, up to $M$ of each species, of which carbon's are the
$\binom{M}{3}^4$ with three nucleons in each species. $H$ never leaves those, since it
keeps each species' number. `weak.nucleus`, a `Nucleus`, counts carbon's nucleons in
each species. Positions take fewer qubits but costlier gates for the walk; they are
where argon lives.

In [17]:
print(f"system qubits: {lattice.system_qubits}")
print(
    f"basis states of the register: 2^{lattice.system_qubits} = "
    f"{2**lattice.system_qubits:.2e}"
)
print(
    f"nucleons per species (p up, p down, n up, n down): {nucleus.nucleons_per_species}"
)
print(f"carbon's basis states among them: {amplitudes:.2e}")

system qubits: 256
basis states of the register: 2^256 = 1.16e+77
nucleons per species (p up, p down, n up, n down): (3, 3, 3, 3)
carbon's basis states among them: 3.01e+18


## From fermions to qubits: the Jordan-Wigner strings

Line the modes up in one order. The occupation of mode $j$ is $n_j = (1 - Z_j)/2$: 0 on
$|0\rangle$, 1 on $|1\rangle$. So every on-site contact, a product of $n$'s, is a product
of $Z$'s. Moving a nucleon needs more care. Fermion operators anticommute,
$\{a_i, a_j^\dagger\} = \delta_{ij}$ and $\{a_i, a_j\} = 0$, while operators on different
qubits commute. Jordan and Wigner attach a $Z$ to every earlier mode:

$$a_j = Z_0 Z_1 \cdots Z_{j-1}\, \frac{X_j + iY_j}{2},$$

where $(X + iY)/2 = |0\rangle\langle 1|$ empties mode $j$, and the $Z$'s give a sign
$-1$ for each occupied mode before it. A hop between modes $i < j$ then becomes two Pauli
strings,

$$a_i^\dagger a_j + a_j^\dagger a_i = \tfrac{1}{2}\left(X_i Z \cdots Z X_j + Y_i Z \cdots Z Y_j\right),$$

with a $Z$ on each mode strictly between. On a 3D lattice, neighbors along $y$ and $z$
sit far apart in the line, so their strings are long; a string's length costs only
Clifford gates. The next cell checks all of this on three modes, and what goes wrong
without the $Z$'s.

In [18]:
modes = cirq.LineQubit.range(3)
annihilators = []
without_strings = []
for j in range(3):
    earlier = cirq.PauliString({modes[k]: cirq.Z for k in range(j)})
    empties = 0.5 * earlier * cirq.X(modes[j]) + 0.5j * earlier * cirq.Y(modes[j])
    annihilators.append(empties.matrix(modes))
    alone = 0.5 * cirq.X(modes[j]) + 0.5j * cirq.Y(modes[j])
    without_strings.append(alone.matrix(modes))
for i, j in itertools.product(range(3), repeat=2):
    a_i, a_j = annihilators[i], annihilators[j]
    delta = np.eye(8) if i == j else np.zeros((8, 8))
    assert np.allclose(a_i @ a_j.conj().T + a_j.conj().T @ a_i, delta)
    assert np.allclose(a_i @ a_j + a_j @ a_i, 0)
occupation = annihilators[1].conj().T @ annihilators[1]
number = 0.5 * cirq.PauliString() - 0.5 * cirq.Z(modes[1])
assert np.allclose(occupation, number.matrix(modes))
hop = (
    annihilators[0].conj().T @ annihilators[2]
    + annihilators[2].conj().T @ annihilators[0]
)
two_strings = 0.5 * cirq.X(modes[0]) * cirq.Z(modes[1]) * cirq.X(
    modes[2]
) + 0.5 * cirq.Y(modes[0]) * cirq.Z(modes[1]) * cirq.Y(modes[2])
assert np.allclose(hop, two_strings.matrix(modes))
print(
    "with the strings: {a_i, a_j^dagger} = delta_ij, {a_i, a_j} = 0, "
    "n_j = (1 - Z_j) / 2"
)
print(f"a hop between modes 0 and 2: {two_strings}")
mixed = (
    without_strings[0] @ without_strings[1].conj().T
    + without_strings[1].conj().T @ without_strings[0]
)
print(
    "without them, {a_0, a_1^dagger} is not 0: "
    f"its largest entry is {np.max(np.abs(mixed)):.0f}"
)

with the strings: {a_i, a_j^dagger} = delta_ij, {a_i, a_j} = 0, n_j = (1 - Z_j) / 2
a hop between modes 0 and 2: 0.5*X(q(0))*Z(q(1))*X(q(2))+0.5*Y(q(0))*Z(q(1))*Y(q(2))
without them, {a_0, a_1^dagger} is not 0: its largest entry is 2


## One site's 38 terms, and $\Gamma = 2{,}432$

How many terms does the Hamiltonian have, once it is written on qubits? Put
$n = (1 - Z)/2$ into the contacts and multiply out. A site's six pairs give
$\frac{1}{4}\left(6 - 3\sum_f Z_f + \sum_{f<f'} Z_f Z_{f'}\right)$, each $Z_f$ appearing in
three pairs. Its four triples give
$\frac{1}{8}\left(4 - 3\sum_f Z_f + 2\sum_{f<f'} Z_f Z_{f'} - \sum_{f<f'<f''} Z_f Z_{f'} Z_{f''}\right)$,
each $Z_f$ in three triples and each pair in two. Collected, a site carries 14 on-site
strings,

| strings | count | coefficient of each |
|---|---|---|
| single $Z$ | 4 | $-\frac{3}{4}(C_0 + D_0/2)$ |
| pairs $ZZ$ | 6 | $\frac{1}{4}(C_0 + D_0)$ |
| triples $ZZZ$ | 4 | $-D_0/8$ |

and a constant, $\frac{3}{2}C_0 + \frac{1}{2}D_0$, which shifts every energy by the same
amount and is dropped. The hops add 24 strings at $-t/2$ each. Each bond's term holds
the hop both ways, $a^\dagger_{i,f} a_{j,f} + a^\dagger_{j,f} a_{i,f}$. The two halves
are each other's Hermitian conjugates, so a Hermitian $H$ has both with the same
coefficient, and the Jordan-Wigner cell above turned the pair into two strings. Each
bond is counted once, at the site on its minus side, so a site has 3 bonds, toward $+x$,
$+y$ and $+z$: 3 directions, 4 species and 2 strings make 24. With the 14 on-site
strings, a site has 38, `TERMS_PER_SITE` (Roggero et al. 2019, App. C.1), and
$\Gamma = 38M$ over the lattice, `Lattice.term_count`. `pauli_coefficients`, from the
library module `norms`, gives each kind's coefficient.

The next cell writes one site's 14 on-site strings for Cirq, with carbon's coefficients:
one site of carbon's lattice, its four species on four qubits. These 14 strings alone
make the one-site example, on which the notebook runs every step of the energy
measurement; "Measuring an energy on a quantum computer" says what it is for.

In [19]:
coefficients = pauli_coefficients(LECTURE_COUPLINGS)
site = [cirq.NamedQubit(name) for name in ("p_up", "p_down", "n_up", "n_down")]
on_site_strings = []
words = []
for size, coefficient in [
    (1, coefficients.single_z_mev),
    (2, coefficients.pair_zz_mev),
    (3, coefficients.triple_zzz_mev),
]:
    for chosen in itertools.combinations(site, size):
        string = cirq.PauliString({qubit: cirq.Z for qubit in chosen})
        on_site_strings.append((coefficient, string))
        words.append("".join("Z" if qubit in chosen else "I" for qubit in site))
print("the 14 strings, on (p up, p down, n up, n down):")
for (coefficient, _), word in zip(on_site_strings, words, strict=True):
    print(f"{coefficient:+7.2f} MeV  {word}")
print(f"the constant, dropped: {coefficients.identity_mev:+.2f} MeV")
print(f"per site: 24 hopping + 4 Z + 6 ZZ + 4 ZZZ = {TERMS_PER_SITE}")
print(f"Gamma, over the lattice: {lattice.term_count:,} terms")

the 14 strings, on (p up, p down, n up, n down):
 +25.73 MeV  ZIII
 +25.73 MeV  IZII
 +25.73 MeV  IIZI
 +25.73 MeV  IIIZ
  +7.40 MeV  ZZII
  +7.40 MeV  ZIZI
  +7.40 MeV  ZIIZ
  +7.40 MeV  IZZI
  +7.40 MeV  IZIZ
  +7.40 MeV  IIZZ
 -15.98 MeV  ZZZI
 -15.98 MeV  ZZIZ
 -15.98 MeV  ZIZZ
 -15.98 MeV  IZZZ
the constant, dropped: -83.42 MeV
per site: 24 hopping + 4 Z + 6 ZZ + 4 ZZZ = 38
Gamma, over the lattice: 2,432 terms


In [20]:
site_hamiltonian = sum(coefficient * string for coefficient, string in on_site_strings)
site_matrix = site_hamiltonian.matrix(site).real
assert np.allclose(site_matrix, np.diag(np.diag(site_matrix)))
for occupation in range(16):
    nucleons = format(occupation, "04b").count("1")
    contact = C0 * math.comb(nucleons, 2) + D0 * math.comb(nucleons, 3)
    energy = site_matrix[occupation, occupation] + coefficients.identity_mev
    assert math.isclose(energy, contact, abs_tol=1e-9)
print("on all 16 occupations of a site, the strings and the constant give")
print("C0 for each pair of nucleons and D0 for each triple, as Eq. 1 does")

on all 16 occupations of a site, the strings and the constant give
C0 for each pair of nucleons and D0 for each triple, as Eq. 1 does


## $\lambda$: the size of the Hamiltonian, in one number

How large is the Hamiltonian? Write $H = \sum_\gamma c_\gamma U_\gamma$, Pauli strings
$U_\gamma$ with real coefficients $c_\gamma$, less the constant. Then
$\lambda = \sum_\gamma |c_\gamma|$, the one-norm of the coefficients, and no energy of
$H$, less its constant, exceeds $\lambda$ in size. Site by site, from the table above,

$$\frac{\lambda}{M} = 24 \cdot \frac{|t|}{2} + 4 \cdot \tfrac{3}{4}\left|C_0 + \tfrac{D_0}{2}\right| + 6 \cdot \tfrac{1}{4}\left|C_0 + D_0\right| + 4 \cdot \frac{|D_0|}{8}.$$

The couplings are those of "The model: nucleons on a lattice", Roggero et al. 2019's
Tab. I, whose contacts were fitted on a richer Hamiltonian. Re-fitting the contacts on
this one, to the deuteron's binding of 2.224 MeV and the triton's energy of $-8.48$ MeV,
gives $C_0 = -93.63$ and $D_0 = +141.6$ MeV and leaves $\lambda$ at 338.2 MeV per site.
That comes from a near cancellation: for these signs one site's on-site share is
$\tfrac{3}{2}|C_0| + D_0/2$, and the re-fit's weaker $C_0$ and stronger $D_0$ move it by
about 6.9 MeV each, in opposite directions. The re-fit comes from a script we have not
released.

Why it matters: a quantum computer can only apply $H/\lambda$, inside a unitary (the
next section), so $\lambda$ sets the scale of every cost of the energy measurement.

Why divide $H$ by $\lambda$? A unitary keeps every vector's length, so a corner of one
can never lengthen a vector, and a Hermitian matrix in that corner has every eigenvalue
between $-1$ and $1$. To fit there, $H$ must be divided by an energy at least as large
as its largest energy in size, and $\lambda$ is one, as the first paragraph says. It is
also the one that the construction from the list of terms gives, since there the
strings' weights $|c_\gamma|/\lambda$ must add to one, like probabilities ("Block
encoding (ii)"). $\lambda$ can exceed the largest energy by far: the one-site example's
$\lambda$ is 211.3 MeV and its largest energy, less the constant, 83.4 MeV, as the next
cell prints. The price comes at the readout: the walk turns an energy $E$ into an angle
that depends on $E/\lambda$ alone, so the larger $\lambda$ is, the more walk steps it
takes to read energies in bins of the same width. "The walk and the readout, priced"
shows what that means as the lattice grows.

`one_site_one_norm()` gives one site's share, group by group, from the coefficients of
Roggero et al. 2019, Eq. 1, at the lecture's couplings. `Lattice.one_norm_mev` gives
$\lambda$ over the lattice, $M$ times one site's share, since every site of a periodic
lattice is alike. The last lines take the one-site example's $\lambda$, its 14 strings
alone.

In [21]:
pieces = one_site_one_norm()
print(f"24 hopping strings: {pieces.hopping_mev:.1f} MeV")
print(f"4 single Z strings: {pieces.single_z_mev:.1f} MeV")
print(f"6 ZZ strings: {pieces.pair_zz_mev:.1f} MeV")
print(f"4 ZZZ strings: {pieces.triple_zzz_mev:.1f} MeV")
print(f"the 14 on-site strings together: {pieces.on_site_mev:.1f} MeV")
print(f"one site: {pieces.total_mev:.1f} MeV")
print(f"lambda, over {lattice.site_count} sites: {lattice.one_norm_mev:,.0f} MeV")
site_lambda = 0.0
for coefficient, _ in on_site_strings:
    site_lambda += abs(coefficient)
assert math.isclose(site_lambda, pieces.on_site_mev)
largest = np.max(np.abs(np.diag(site_matrix)))
print(f"the one-site example: lambda = {site_lambda:.1f} MeV, and its largest energy,")
print(f"less the constant, is {largest:.1f} MeV in size")

24 hopping strings: 127.0 MeV
4 single Z strings: 102.9 MeV
6 ZZ strings: 44.4 MeV
4 ZZZ strings: 63.9 MeV
the 14 on-site strings together: 211.3 MeV
one site: 338.2 MeV
lambda, over 64 sites: 21,646 MeV
the one-site example: lambda = 211.3 MeV, and its largest energy,
less the constant, is 83.4 MeV in size


# Measuring an energy on a quantum computer

How does a quantum computer measure an energy? The next sections build the measurement
in three parts: a block encoding of $H$, the walk made from it, and phase estimation,
which reads an energy off the walk. Each part rests on a claim about matrices: that a
corner of a unitary holds $H/\lambda$, that the walk turns each energy eigenstate by an
angle its energy sets, that phase estimation returns that angle. Carbon's matrices act
on 256 system qubits, $2^{256}$ amplitudes, which no computer can write down, so
carbon's circuits can only be counted and priced. To watch each claim hold, the notebook
runs the same steps on a Hamiltonian small enough to compute exactly, the one-site
example.

The example is one site of carbon's lattice: its four modes, $p\!\uparrow$,
$p\!\downarrow$, $n\!\uparrow$ and $n\!\downarrow$, on four qubits, with only its 14
on-site strings at carbon's coefficients ("One site's 38 terms" wrote them). It charges
$C_0$ for each pair of nucleons on the site and $D_0$ for each triple, so its 16
energies are known by counting:

| nucleons on the site | occupations | energy |
|---|---|---|
| 0 | 1 | 0 |
| 1 | 4 | 0 |
| 2 | 6 | $C_0$ |
| 3 | 4 | $3C_0 + D_0$ |
| 4 | 1 | $6C_0 + 4D_0$ |

The strings leave out the constant, $\frac{3}{2}C_0 + \frac{1}{2}D_0$, so the circuits
see each energy less it; the next cell prints those.

Why this one? It is small: four system qubits, four ancillas to index its 14 strings,
and 15 qubits at most once phase estimation adds its register. So Cirq computes every
matrix, and the claims are checked by an `assert` as they come. Its answers are known:
its $H$ is diagonal, every occupation an energy eigenstate, so whatever a circuit
returns can be held against the table. And it is carbon's in miniature: the same kinds
of strings at the same coefficients, with a SELECT and a REFLECT built gate by gate
and a PREPARE applied as a matrix.

What it leaves out: the hops, so no nucleon moves, which is what makes its $H$ diagonal
and its answers easy; the size, 14 strings where carbon has 2,432 (SELECT, a tree over
all the strings, is 92% of carbon's walk step, "One walk step, priced"); and the hard
part, carbon's ground state, which the example never needs, since its eigenstates are
its occupations. Carbon's counts follow in each section, the same constructions at full
size, counted and priced. The example runs from here to "Phase estimation (iii)". It
returns in "Loading a known state". Two later sections, "Applying the current" and
"Amplitude amplification", leave it for a ring of four sites, because one site has no
momentum to transfer.

In [22]:
energies_by_nucleons = {}
for occupation in range(16):
    nucleons = format(occupation, "04b").count("1")
    energy = site_matrix[occupation, occupation]
    energies_by_nucleons.setdefault(nucleons, []).append(energy)
print("the one-site example's energies, less the constant, by nucleons on the site:")
for nucleons, energies in sorted(energies_by_nucleons.items()):
    assert np.allclose(energies, energies[0])
    count = len(energies)
    print(f"  {nucleons}: {energies[0]:+.3f} MeV, {count} of the 16 occupations")

the one-site example's energies, less the constant, by nucleons on the site:
  0: +83.425 MeV, 1 of the 16 occupations
  1: +83.425 MeV, 4 of the 16 occupations
  2: -14.805 MeV, 6 of the 16 occupations
  3: -83.425 MeV, 4 of the 16 occupations
  4: +5.405 MeV, 1 of the 16 occupations


## Block encoding (i): applying $H$ with unitaries

A quantum computer applies only unitaries, and $H$ is not one. But
$H = \sum_\gamma c_\gamma U_\gamma$ is a list of Pauli strings, and each string
$U_\gamma$ is a unitary the machine can apply. The plan is to apply every string at
once, in superposition, each weighted by its coefficient, with a few extra qubits, an
ancilla register, holding the weights, and then to recombine. This is a linear
combination of unitaries (Babbush et al., arXiv:1805.03662).

The weights must be probabilities: $p_\gamma = |c_\gamma|/\lambda$, which add up to one
because $\lambda = \sum_\gamma |c_\gamma|$ ("$\lambda$: the size of the Hamiltonian").
Each coefficient's sign is folded into its $U_\gamma$. So
$H = \lambda \sum_\gamma p_\gamma U_\gamma$: $H/\lambda$ is a weighted average of
unitaries, and the list of terms alone fixes $\lambda$.

Three steps do it, each a unitary: PREPARE writes the weights onto the ancillas, SELECT
applies $U_\gamma$ when the ancillas read $\gamma$, and PREPARE undone recombines.
Together they make one unitary, $B$, whose matrix holds $H/\lambda$ in one block: a
block encoding of $H$.

## Block encoding (ii): PREPARE and SELECT

PREPARE writes the weights, as amplitudes, onto an ancilla register of about
$\log_2 \Gamma$ qubits, $|0\rangle \to \sum_\gamma \sqrt{p_\gamma}\,|\gamma\rangle$. The
amplitudes are square roots, so that reading the register would give $\gamma$ with
probability $p_\gamma$. SELECT applies $U_\gamma$ to the system when the ancilla
register reads $\gamma$, $\sum_\gamma |\gamma\rangle\langle\gamma| \otimes U_\gamma$.
On the superposition PREPARE made, it applies every string at once, each in its own
branch.

PREPARE and SELECT are the block encoding's oracles: subroutines the algorithm calls
and counts. Each is a large circuit for carbon, SELECT alone a tree of $\Gamma - 1$
ANDs, so the slides draw neither. "PREPARE" and "SELECT" below give their constructions
and prices, the next two cells build them for the example, and the module's `walk_step`
counts carbon's gates from the published constructions (Babbush et al., Sec. III).
Qualtran builds both down to gates (arXiv:2409.04643), for anyone who wants to go all
the way; it is not installed with this notebook.

For the example, four ancilla qubits index the 14 strings, two of their 16 values
unused. PREPARE can be any unitary whose first column holds the $\sqrt{p_\gamma}$. The
next cell takes the reflection that swaps $|0\rangle$ with that column, a Householder
reflection, and Cirq applies it as a matrix; carbon's PREPARE is built from gates, by
alias sampling ("PREPARE" below). SELECT is built gate by gate: for each value of the
index, a $Z$ on each mode of the string, controlled on the index reading that value, and
a sign where the coefficient is negative.

In [23]:
index = [cirq.NamedQubit(f"index{bit}") for bit in range(4)]
weights = np.zeros(16)
for gamma, (coefficient, _) in enumerate(on_site_strings):
    weights[gamma] = abs(coefficient) / site_lambda
mirror = np.eye(16)[0] - np.sqrt(weights)
prepare_matrix = np.eye(16) - 2 * np.outer(mirror, mirror) / (mirror @ mirror)
assert np.allclose(prepare_matrix[:, 0], np.sqrt(weights))
prepare = cirq.MatrixGate(prepare_matrix, name="PREPARE")
unprepare = cirq.MatrixGate(prepare_matrix.conj().T, name="PREPARE†")
print(f"PREPARE: |0> -> the amplitudes sqrt(p) of {len(on_site_strings)} strings")

PREPARE: |0> -> the amplitudes sqrt(p) of 14 strings


In [24]:
select = cirq.Circuit()
for gamma, (coefficient, string) in enumerate(on_site_strings):
    reads = [int(bit) for bit in format(gamma, "04b")]
    for qubit in string.qubits:
        select.append(cirq.Z(qubit).controlled_by(*index, control_values=reads))
    if coefficient < 0:
        sign = cirq.GlobalPhaseGate(-1).on()
        select.append(sign.controlled_by(*index, control_values=reads))
select_gate = cirq.MatrixGate(
    select.unitary(qubit_order=[*index, *site]), name="SELECT"
)
block = cirq.Circuit(
    prepare.on(*index), select_gate.on(*index, *site), unprepare.on(*index)
)
print(block.to_text_diagram(qubit_order=[*index, *site]))

index0: ───PREPARE[1]───SELECT[1]───PREPARE†[1]───
           │            │           │
index1: ───PREPARE[2]───SELECT[2]───PREPARE†[2]───
           │            │           │
index2: ───PREPARE[3]───SELECT[3]───PREPARE†[3]───
           │            │           │
index3: ───PREPARE[4]───SELECT[4]───PREPARE†[4]───
                        │
p_up: ──────────────────SELECT[5]─────────────────
                        │
p_down: ────────────────SELECT[6]─────────────────
                        │
n_up: ──────────────────SELECT[7]─────────────────
                        │
n_down: ────────────────SELECT[8]─────────────────


## Block encoding (iii): following a state through them

Follow $|0\rangle|\psi\rangle$ through PREPARE, SELECT and PREPARE undone; the last cell
drew them for the example:

$$\begin{aligned} |0\rangle|\psi\rangle &\xrightarrow{\ \mathrm{PREPARE}\ } \sum_\gamma \sqrt{p_\gamma}\,|\gamma\rangle|\psi\rangle \\ &\xrightarrow{\ \mathrm{SELECT}\ } \sum_\gamma \sqrt{p_\gamma}\,|\gamma\rangle\,U_\gamma|\psi\rangle \\ &\xrightarrow{\ \mathrm{PREPARE}^\dagger\ } |0\rangle \sum_\gamma \sqrt{p_\gamma}\sqrt{p_\gamma}\; U_\gamma|\psi\rangle + s\,|{\perp}\rangle = |0\rangle\,(H/\lambda)|\psi\rangle + s\,|{\perp}\rangle. \end{aligned}$$

After SELECT, each branch $|\gamma\rangle$ of the ancilla register holds a different
state, $U_\gamma|\psi\rangle$. PREPARE undone sends each $|\gamma\rangle$ back to
$|0\rangle$ with amplitude $\sqrt{p_\gamma}$, the amplitude PREPARE gave it, so each
term carries $\sqrt{p_\gamma}$ twice. The branches return to $|0\rangle$ only as far as
they agree: the part that returns is their weighted average, $(H/\lambda)|\psi\rangle$.
The rest, $s\,|{\perp}\rangle$, is the leftover: $|{\perp}\rangle$ is a state of
ancillas and system together, of length one, with the ancillas away from $|0\rangle$.
Had every string turned $|\psi\rangle$ into the same state, all of it would return, with
no leftover.

The next cell checks this on the example, for two nucleons on the site,
$p\!\uparrow$ and $p\!\downarrow$: the part with the ancillas back in $|0\rangle$, and
the leftover's share.

In [25]:
block_matrix = block.unitary(qubit_order=[*index, *site])
assert np.allclose(block_matrix[:16, :16], site_matrix / site_lambda)
pair = int("1100", 2)
entering = np.eye(256)[pair]
leaving = block_matrix @ entering
ancillas_read_zero = np.sum(np.abs(leaving[:16]) ** 2)
leftover = np.sum(np.abs(leaving[16:]) ** 2)
pair_energy = site_matrix[pair, pair]
assert math.isclose(ancillas_read_zero, (pair_energy / site_lambda) ** 2)
assert math.isclose(ancillas_read_zero + leftover, 1)
print("the corner, ancillas in |0> on the way in and out, is H / lambda")
print(
    f"two nucleons, p up and p down: C0 = {C0} MeV; less the constant, "
    f"the energy the circuit sees, E = {pair_energy:.2f} MeV"
)
print(
    "measured, the ancillas read 0 with probability (E / lambda)^2 = "
    f"{ancillas_read_zero:.4f}"
)
print(f"the leftover carries the rest: s^2 = 1 - (E / lambda)^2 = {leftover:.4f}")

the corner, ancillas in |0> on the way in and out, is H / lambda
two nucleons, p up and p down: C0 = -98.23 MeV; less the constant, the energy the circuit sees, E = -14.80 MeV
measured, the ancillas read 0 with probability (E / lambda)^2 = 0.0049
the leftover carries the rest: s^2 = 1 - (E / lambda)^2 = 0.9951


## Block encoding (iv): a matrix hidden inside a unitary

The three steps make one unitary, $B$, on the system and the ancillas. With its rows and
columns sorted by the ancillas' state, its matrix holds $H/\lambda$ in one block, where
the ancillas read 0 on the way in and on the way out, and the block below it makes the
leftover: hence the name. The rule: with the ancillas in $|0\rangle$, their all-zero
state, and the system in any state $|\psi\rangle$,

$$B\,|0\rangle|\psi\rangle = |0\rangle\,(H/\lambda)|\psi\rangle + s\,|{\perp}\rangle.$$

$B$ keeps lengths, so $\|H|\psi\rangle\|^2/\lambda^2 + s^2 = 1$: nothing vanishes, and
the leftover carries the rest. If the ancillas are then measured and every one reads 0,
the measurement has thrown the leftover away, and the system is left holding
$H|\psi\rangle$, normalized. That happens with probability
$\|H|\psi\rangle\|^2/\lambda^2$, so a large $\lambda$ makes success rare.

Papers write the rule in a compact form that hides the leftover:

$$(\langle 0| \otimes I)\; B \;(|0\rangle \otimes I) = H/\lambda.$$

Read it right to left. $|0\rangle \otimes I$ puts the ancillas in $|0\rangle$ and
leaves the system alone: $I$ is the identity on the system, so the rule holds for every
$|\psi\rangle$ at once. Then $B$ acts. Then $\langle 0| \otimes I$ keeps only the part
of the result whose ancillas read 0: it throws the leftover away. What remains is an
operator on the system alone, $H/\lambda$, the block of $B$ whose ancillas are
$|0\rangle$ on the way in and on the way out. The equation says nothing about the rest
of $B$, which is still there and makes the leftover. Papers often shorten the form
further, writing $\langle 0|$ for $\langle 0| \otimes I$ and $|0\rangle$ for
$|0\rangle \otimes I$. Babbush et al. fold PREPARE into the state
$|L\rangle = \mathrm{PREPARE}\,|0\rangle$ and write
$(\langle L| \otimes I)\,\mathrm{SELECT}\,(|L\rangle \otimes I) = H/\lambda$ (their
Eq. 8).

Why $\lambda$ sets the energy measurement's cost: the walk made from this block encoding
turns each energy eigenstate by an angle $\theta_E$ with $\cos\theta_E = E/\lambda$
("Why the walk encodes the energy"), and phase estimation reads that angle. $W_q$ bits
read energies in bins of $2\pi\lambda/2^{W_q}$ with $2^{W_q - 1}$ walk steps ("Phase
estimation (ii)"), so bins of width $\delta E$ take about $\pi\lambda/\delta E$ steps,
and doubling $\lambda$ doubles the steps for the same bins. A preparation of the ground
state from $H$ pays the same way: projection takes about $\lambda/(\gamma\Delta)$ uses
of the block encoding ("Why carbon's preparation is still unpriced"). The current is
block-encoded separately, with its own normalization $\Lambda_J$, and a loaded table of
amplitudes never uses $H$, so $\lambda$ does not enter their costs.

## Block encoding's two uses: throw the leftover away, or keep it

The recipe uses two block encodings, in opposite ways: the current's in step 2, and
$H$'s in step 3.

Step 2, the current $J$: throw the leftover away. $J$ has a block encoding of its own,
built the same way from its own terms ("Applying the current" below). We want one state
from it, $J|\Psi_0\rangle$, normalized: phase estimation samples its energies, and their
weights are the response. The leftover's energies are not the response, so it must go.
The ancillas are measured once, before step 3, and the run is kept only if every one
reads 0. Unaided, that is rare for carbon ("Applying the current" prints the odds).
Amplitude amplification first turns the state toward success; its rounds are unitary,
so they keep the leftover until that one measurement ("Amplitude amplification" below).

Step 3, the walk: keep the leftover. Phase estimation reads energies from a unitary
whose eigenvalues carry them, and for carbon it uses $H$'s block encoding $B$ thousands
of times ("Phase estimation (ii)"). If $B$ were measured after each use, each use would
succeed with probability $\|H|\psi\rangle\|^2/\lambda^2$, and the chances would
multiply; the next cell prints them for the example's proton pair. Unmeasured, $B$
followed by REFLECT, a sign flip on the ancillas, is the walk, a unitary: each step
turns $|0\rangle|\psi_E\rangle$ by an angle set by $E/\lambda$, in the plane it shares
with its leftover ("Why the walk encodes the energy"). Nothing is thrown away, no step
fails, and phase estimation reads the angle.

In [26]:
print("uses of B, each measured: the chance that every one reads 0")
for uses in (1, 2, 5, 10):
    print(f"{uses:2d}: {ancillas_read_zero**uses:.1e}")

uses of B, each measured: the chance that every one reads 0
 1: 4.9e-03
 2: 2.4e-05
 5: 2.9e-12
10: 8.2e-24


## The walk: the block encoding, then one reflection

One step of the walk acts on the system and the ancilla register: PREPARE, SELECT,
PREPARE$^\dagger$, the block encoding $B$, and then REFLECT, the
reflection $R = 2|0\rangle\langle 0| - I$ on the ancilla register. $R$ leaves the
all-zero ancilla state alone and flips the sign of every other. The walk is this step
repeated, without measuring the ancillas. Each step turns every energy eigenstate by the
same angle, and the angle encodes the energy ("Why the walk encodes the energy" below).
The next cell builds a step for the example. Its REFLECT applies the sign flip as one $Z$
controlled by the other three ancillas; "REFLECT" below opens that box.

In [27]:
reflect = cirq.Circuit(
    [cirq.X(qubit) for qubit in index],
    cirq.Z(index[3]).controlled_by(*index[:3]),
    [cirq.X(qubit) for qubit in index],
    cirq.global_phase_operation(-1),
)
reflect_matrix = reflect.unitary(qubit_order=index)
zero = np.eye(16)[0]
assert np.allclose(reflect_matrix, 2 * np.outer(zero, zero) - np.eye(16))
reflect_gate = cirq.MatrixGate(reflect_matrix, name="REFLECT")
walk = cirq.Circuit(
    prepare.on(*index),
    select_gate.on(*index, *site),
    unprepare.on(*index),
    reflect_gate.on(*index),
)
walk_matrix = walk.unitary(qubit_order=[*index, *site])
print(walk.to_text_diagram(qubit_order=[*index, *site]))

index0: ───PREPARE[1]───SELECT[1]───PREPARE†[1]───REFLECT[1]───
           │            │           │             │
index1: ───PREPARE[2]───SELECT[2]───PREPARE†[2]───REFLECT[2]───
           │            │           │             │
index2: ───PREPARE[3]───SELECT[3]───PREPARE†[3]───REFLECT[3]───
           │            │           │             │
index3: ───PREPARE[4]───SELECT[4]───PREPARE†[4]───REFLECT[4]───
                        │
p_up: ──────────────────SELECT[5]──────────────────────────────
                        │
p_down: ────────────────SELECT[6]──────────────────────────────
                        │
n_up: ──────────────────SELECT[7]──────────────────────────────
                        │
n_down: ────────────────SELECT[8]──────────────────────────────


## PREPARE: the table of coefficients, from one site

PREPARE's job is to turn $|0\rangle$ into $\sum_\gamma \sqrt{p_\gamma}\,|\gamma\rangle$.
For carbon that is $\Gamma$ entries, but every site has the same 38 coefficients, so the
ancilla register splits into a site index, in an equal superposition over the $M$
sites, and a term index holding one site's 38 amplitudes (Babbush et al., Figs. 11-12).
For the example it is one site's 14 weights.

One site's amplitudes are made by alias sampling (ibid., Sec. III D). It is easiest to
see first as a game of chance, with no qubits. Roll a fair die with one face per entry,
then flip a coin biased for the face that came up: heads keeps the face, tails returns
the face's alternate, another entry. With the right keep chance and alternate for each
face, the game returns every entry with its weight. To find them, draw the weights as
bars, one column per entry, each column to end exactly as tall as the average, $1/L$
for $L$ entries. Take a bar that is too short, fill its column from a bar that is too
tall, and record the tall one as the short one's alternate. That column is finished,
and the tall bar is shorter by what it gave; if that leaves it short, it waits for a
filler of its own. Every move finishes one column, so the process ends, with each
column holding at most two entries: its own, whose share is the keep chance, and its
alternate's (ibid., Fig. 13).

The machine stores each keep chance as a whole number of steps of $1/2^\mu$, the keep
number, on $\mu$ bits. So the weights are first rounded to squares of $1/(2^\mu L)$,
$2^\mu$ squares to a column. The next cell does this for the example with $\mu = 3$, as
in Babbush et al.'s Fig. 13: 14 columns of 8 squares. It prints each entry's weight,
its squares, its keep number (the squares of its own that stay in its column) and its
alternate.

In [28]:
entries = len(on_site_strings)
keep_bits_example = 3
column_squares = 2**keep_bits_example
total_squares = entries * column_squares
exact = weights[:entries] * total_squares
squares = np.floor(exact).astype(int)
remainders = exact - squares
squares[np.argsort(-remainders)[: total_squares - squares.sum()]] += 1

keep = np.full(entries, column_squares)
alternate = np.arange(entries)
squares_left = squares.copy()
short = [entry for entry in range(entries) if squares_left[entry] < column_squares]
tall = [entry for entry in range(entries) if squares_left[entry] > column_squares]
while short:
    low = short.pop()
    high = tall[-1]
    keep[low] = squares_left[low]
    alternate[low] = high
    squares_left[high] -= column_squares - squares_left[low]
    if squares_left[high] < column_squares:
        short.append(tall.pop())
    elif squares_left[high] == column_squares:
        tall.pop()

print(f"{entries} columns of {column_squares} squares, {total_squares} squares in all")
print("index      U      p       squares  keep  alternate")
for entry, (coefficient, _) in enumerate(on_site_strings):
    sign = "+" if coefficient > 0 else "-"
    partner = alternate[entry] if keep[entry] < column_squares else "-"
    print(
        f"{entry:2d} = {entry:04b}  {sign}{words[entry]}  {weights[entry]:.4f}  "
        f"{squares[entry]:7d}  {keep[entry]:4d}  {partner:>9}"
    )

14 columns of 8 squares, 112 squares in all
index      U      p       squares  keep  alternate
 0 = 0000  +ZIII  0.1218       14     8          -
 1 = 0001  +IZII  0.1218       14     6          0
 2 = 0010  +IIZI  0.1218       14     8          -
 3 = 0011  +IIIZ  0.1218       14     6          2
 4 = 0100  +ZZII  0.0350        4     4          0
 5 = 0101  +ZIZI  0.0350        4     4          1
 6 = 0110  +ZIIZ  0.0350        4     4          1
 7 = 0111  +IZZI  0.0350        4     4          2
 8 = 1000  +IZIZ  0.0350        4     4          3
 9 = 1001  +IIZZ  0.0350        4     4          3
10 = 1010  -ZZZI  0.0756        8     8          -
11 = 1011  -ZZIZ  0.0756        8     8          -
12 = 1100  -ZIZZ  0.0756        8     8          -
13 = 1101  -IZZZ  0.0756        8     8          -


The circuit plays the same game on every path at once (Babbush et al., Fig. 11). An
equal superposition over the $L$ entries is the die. A table lookup writes each entry's
keep number and alternate into ancillas. The coin is a register $\sigma$ of $\mu$ qubits
in an equal superposition over its $2^\mu$ values, compared with the keep number: where
$\sigma$ is not below it, the index and the alternate are swapped. Each of the
$2^\mu L$ paths, one entry and one value of $\sigma$, has the same amplitude and ends on
one entry, so an entry ends with as many paths as it has squares.

Why do the paths add, rather than interfere? Two paths that end on the same entry leave
different values in the ancillas, $\sigma$ and the looked-up data, so their states are
orthogonal and their probabilities add. PREPARE's state is therefore not
$\sum_\gamma \sqrt{p_\gamma}\,|\gamma\rangle$ alone but
$\sum_\gamma \sqrt{p_\gamma}\,|\gamma\rangle|\mathrm{junk}_\gamma\rangle$: the index
entangled with what the ancillas hold. That is enough for the block encoding. SELECT
reads only the index, so the junk passes through it untouched, and PREPARE undone clears
it from the part that returns to $|0\rangle$ (ibid., Eqs. 33-34). The next cell follows
all of the example's paths and checks that each entry ends on its squares, so that each
weight comes out within one square, $1/(2^\mu L)$, of its target (ibid., Eq. 35).

In [29]:
endings = np.zeros(entries, dtype=int)
for entry in range(entries):
    for sigma in range(column_squares):
        if sigma < keep[entry]:
            endings[entry] += 1
        else:
            endings[alternate[entry]] += 1
assert np.array_equal(endings, squares)
worst = np.max(np.abs(endings / total_squares - weights[:entries]))
assert worst < 1 / total_squares
print(f"{entries} entries x {column_squares} values of sigma = {total_squares} paths")
print("each entry ends on as many paths as it has squares")
print(
    f"the worst weight is off by {worst:.4f}, "
    f"within one square, 1/{total_squares} = {1 / total_squares:.4f}"
)

14 entries x 8 values of sigma = 112 paths
each entry ends on as many paths as it has squares
the worst weight is off by 0.0042, within one square, 1/112 = 0.0089


For carbon, $L = 38$, and the keep number takes more bits, set from the register's
delivered error and a bound on the energies it reads (`lookup_keep_probability_bits`,
which `walk_step` calls). The next cell prints them and takes one PREPARE's price apart.
The table lookup costs one AND per entry but one, the comparison one per keep bit, and
the swap one per qubit of the index (Babbush et al., Figs. 10-11). An equal
superposition costs nothing over a power of two, as the 64 sites are; 38 is twice 19,
and an equal superposition over an odd number of values takes a round of amplitude
amplification, with two rotations (ibid., Fig. 12). `weak.step`, the `WalkStep` that
`walk_step` counts, holds the total.

In [30]:
step = weak.step
entry_superposition = equal_superposition_cost(TERMS_PER_SITE)
site_superposition = equal_superposition_cost(weak.lattice.site_count)
prepare_parts = {
    "the table lookup, one per entry but one": TERMS_PER_SITE - 1,
    "the comparison, one per keep bit": step.keep_bits,
    "the swap, one per index qubit": entry_superposition.register_qubits,
    "the equal superposition over the entries": entry_superposition.toffoli_count,
    "the equal superposition over the sites": site_superposition.toffoli_count,
}
assert sum(prepare_parts.values()) == step.prepare_toffolis
assert (
    entry_superposition.rotation_count + site_superposition.rotation_count
    == step.prepare_rotations
)
print(
    f"carbon's keep number: {step.keep_bits} bits, so each weight is within "
    f"1/({TERMS_PER_SITE} x 2^{step.keep_bits}) = "
    f"{1 / (TERMS_PER_SITE * 2**step.keep_bits):.1e}"
)
for part, toffolis in prepare_parts.items():
    print(f"{toffolis:3d} Toffolis: {part}")
print(
    f"{step.prepare_toffolis:3d} Toffolis in all, and "
    f"{step.prepare_rotations} rotations "
    f"({entry_superposition.rotation_count} from the entries' superposition)"
)
print(
    f"carbon's PREPARE and its inverse: 2 x {step.prepare_toffolis} Toffolis and "
    f"2 x {step.prepare_rotations} rotations = {weak.prepare_pair_t:,} T per step"
)

carbon's keep number: 12 bits, so each weight is within 1/(38 x 2^12) = 6.4e-06
 37 Toffolis: the table lookup, one per entry but one
 12 Toffolis: the comparison, one per keep bit
  6 Toffolis: the swap, one per index qubit
 10 Toffolis: the equal superposition over the entries
  0 Toffolis: the equal superposition over the sites
 65 Toffolis in all, and 2 rotations (2 from the entries' superposition)
carbon's PREPARE and its inverse: 2 x 65 Toffolis and 2 x 2 rotations = 656 T per step


## SELECT: applying the term the ancilla register names

SELECT's job is to apply the Pauli string $U_\gamma$ to the system when the ancilla
register reads $\gamma$. The example's SELECT, built above, does it the plain way: one
string for each of its 14 values, each controlled on all four index qubits. The next
cell draws the last, index 13, which applies $-IZZZ$, a $-ZZZ$ on $p\!\downarrow$,
$n\!\uparrow$ and $n\!\downarrow$. Cirq draws a control that must read 0 as `(0)`. It
draws the sign, a phase of $-1$ applied only when the index reads 13, as a $Z$ on one
index qubit controlled by the others, which is the same thing.

The plain way is costly. A string controlled on several qubits first combines them into
one flag by an AND chain ("Circuit words for hour 2 (ii)"). Phase estimation adds a
control qubit to every walk step, so each value's string has its index qubits and that
control to combine: one AND per index qubit. The next cell counts it for the example.

In [31]:
last_entry = cirq.Circuit(list(select.all_operations())[-4:])
print(last_entry.to_text_diagram(qubit_order=[*index, *site]))
plain_ands = len(on_site_strings) * len(index)
print(
    f"the plain way: {len(on_site_strings)} values x {len(index)} ANDs "
    f"= {plain_ands} ANDs"
)

index0: ───@─────@─────@─────@─────
           │     │     │     │
index1: ───@─────@─────@─────@─────
           │     │     │     │
index2: ───(0)───(0)───(0)───(0)───
           │     │     │     │
index3: ───@─────@─────@─────Z─────
           │     │     │
p_up: ─────┼─────┼─────┼───────────
           │     │     │
p_down: ───Z─────┼─────┼───────────
                 │     │
n_up: ───────────Z─────┼───────────
                       │
n_down: ───────────────Z───────────
the plain way: 14 values x 4 ANDs = 56 ANDs


Unary iteration (Babbush et al., Sec. III A) shares that work. Put the values at the
leaves of a binary tree that branches on one index qubit per level, the top bit first,
and give each level a flag qubit. A flag reads $|1\rangle$ when the control is on and
the index qubits so far match the branch being visited, and the last level's flag
controls the leaf's string. The tree visits the leaves in order:

- Going down a level, the new flag is the AND of the flag above and the next index qubit
  reading 0, the left branch: one AND.
- Moving to the right branch, where that index qubit reads 1, takes no AND. Left and
  right split the flag above between them, so a CNOT from the flag above turns the left
  flag into the right one.
- Two neighboring leaves share every flag above the level where they part, so only the
  flags below it are undone and made again.

So each node with two branches pays one AND, and a tree with $\Gamma$ leaves has
$\Gamma - 1$ of them. Where the values run out, a node has one branch, and its flag
copies the flag above with a CNOT. That shortcut sends the index values the table never
uses, 14 and 15 here, to a wrong leaf, which does no harm: PREPARE never writes them.
Undoing an AND is free on the machine ("Circuit words for hour 2 (i)"); here Cirq undoes
each by running it again.

The next cell builds the tree for the example, with a control qubit and one flag per
level, and draws its first two leaves. It checks that the tree applies the same string
as the plain SELECT for every index value the table uses and every state of the site,
and nothing with the control off, and it counts the tree's ANDs.

In [32]:
select_control = cirq.NamedQubit("control")
levels = len(index)
flags = [cirq.NamedQubit(f"flag{level}") for level in range(levels)]
leaves = len(on_site_strings)
unary_ops = []
held = [[] for _ in range(levels)]
tree_ands = 0
previous_bits = None
for gamma, (coefficient, string) in enumerate(on_site_strings):
    leaf_bits = [int(bit) for bit in format(gamma, "04b")]
    first_new = 0
    if previous_bits is not None:
        parting_level = next(
            level for level in range(levels) if leaf_bits[level] != previous_bits[level]
        )
        for level in reversed(range(parting_level + 1, levels)):
            unary_ops.extend(held[level])
        parent_flag = select_control if parting_level == 0 else flags[parting_level - 1]
        unary_ops.append(cirq.CNOT(parent_flag, flags[parting_level]))
        held[parting_level] = [
            cirq.X(flags[parting_level]).controlled_by(
                parent_flag, index[parting_level]
            )
        ]
        first_new = parting_level + 1
    for level in range(first_new, levels):
        assert leaf_bits[level] == 0
        parent_flag = select_control if level == 0 else flags[level - 1]
        right_branch = leaf_bits[:level] + [1] + [0] * (levels - level - 1)
        if int("".join(str(bit) for bit in right_branch), 2) >= leaves:
            held[level] = [cirq.CNOT(parent_flag, flags[level])]
        else:
            held[level] = [
                cirq.X(flags[level]).controlled_by(
                    parent_flag, index[level], control_values=[1, 0]
                )
            ]
            tree_ands += 1
        unary_ops.extend(held[level])
    for qubit in string.qubits:
        unary_ops.append(cirq.CZ(flags[-1], qubit))
    if coefficient < 0:
        unary_ops.append(cirq.Z(flags[-1]))
    if gamma == 1:
        first_two_leaves = cirq.Circuit(unary_ops)
    previous_bits = leaf_bits
for level in reversed(range(levels)):
    unary_ops.extend(held[level])
unary = cirq.Circuit(unary_ops)

unary_qubits = [select_control, *index, *flags, *site]
print(first_two_leaves.to_text_diagram(qubit_order=unary_qubits))

plain_matrix = cirq.unitary(select_gate)
simulator = cirq.Simulator()
for control_bit in (0, 1):
    for gamma in range(leaves):
        for site_state in range(16):
            start_state = (control_bit << 12) | (gamma << 8) | site_state
            unary_final = simulator.simulate(
                unary, qubit_order=unary_qubits, initial_state=start_state
            ).final_state_vector
            plain_phase = plain_matrix[gamma * 16 + site_state, gamma * 16 + site_state]
            assert np.isclose(
                unary_final[start_state], plain_phase if control_bit else 1
            )
assert tree_ands == leaves - 1 == unary_iteration_select_toffoli_count(leaves)
print("the same strings as the plain SELECT, and nothing with the control off")
print(f"the tree: {tree_ands} ANDs, one per node with two branches")
carbon_plain = weak.lattice.term_count * math.ceil(math.log2(weak.lattice.term_count))
print(
    f"carbon: {step.select_toffolis:,} ANDs = {weak.select_t:,} T, "
    f"{100 * weak.select_share:.0f}% of a walk step; "
    f"the plain way would take {carbon_plain:,}"
)

control: ───@───────────────────────────────────
            │
index0: ────(0)─────────────────────────────────
            │
index1: ────┼─────(0)───────────────────────────
            │     │
index2: ────┼─────┼─────(0)─────────────────────
            │     │     │
index3: ────┼─────┼─────┼─────(0)───────────────
            │     │     │     │
flag0: ─────X─────@─────┼─────┼─────────────────
                  │     │     │
flag1: ───────────X─────@─────┼─────────────────
                        │     │
flag2: ─────────────────X─────@─────────@───────
                              │         │
flag3: ───────────────────────X─────@───X───@───
                                    │       │
p_up: ──────────────────────────────@───────┼───
                                            │
p_down: ────────────────────────────────────@───

n_up: ──────────────────────────────────────────

n_down: ────────────────────────────────────────
the same strings as the plain SELECT, and nothing with th

## REFLECT: the sign flip that makes the walk

REFLECT's job is $R = 2|0\rangle\langle 0| - I$ on the ancilla register. Flip every
ancilla with $X$, so that "all zero" becomes "all one"; apply a $Z$ conditioned on all
of them, by one AND chain; undo the $X$'s. A $Z$ conditioned on $k$ qubits takes $k - 2$
ANDs: the chain over $k - 1$ of them, then a CZ between its last ancilla and the last
qubit. That flips the sign of the all-zero state alone, which is $-R$, and a global
phase of $-1$ makes it $R$. Alone, a global phase changes nothing; in phase estimation,
where the walk is controlled, it becomes a $Z$ on the control qubit, a Clifford gate.

Why it is there: repeating the block encoding alone builds up nothing useful. With the
reflection between repeats, every step is the same rotation, and its angle is the
energy (next section).

The next cell builds the example's REFLECT on its four ancillas this way, with its ANDs
drawn as Toffolis onto two chain qubits. Uncomputing the chain is drawn as the same
Toffolis again, which a machine replaces by the free erasure. The cell after checks it
against the $R$ of the walk, with the chain qubits in $|00\rangle$ on the way in and
out. In phase estimation the walk runs controlled on a bit of the energy register,
which is one more control.

In [33]:
chain = [cirq.NamedQubit("chain0"), cirq.NamedQubit("chain1")]
reflect_by_chain = cirq.Circuit(
    cirq.Moment(cirq.X(qubit) for qubit in index),
    cirq.Moment(cirq.TOFFOLI(index[0], index[1], chain[0])),
    cirq.Moment(cirq.TOFFOLI(chain[0], index[2], chain[1])),
    cirq.Moment(cirq.CZ(chain[1], index[3])),
    cirq.Moment(cirq.TOFFOLI(chain[0], index[2], chain[1])),
    cirq.Moment(cirq.TOFFOLI(index[0], index[1], chain[0])),
    cirq.Moment(cirq.X(qubit) for qubit in index),
    cirq.Moment(cirq.global_phase_operation(-1)),
)
print(reflect_by_chain.to_text_diagram(qubit_order=[*index, *chain]))

index0: ────────X───@───────────────@───X───────
                    │               │
index1: ────────X───@───────────────@───X───────
                    │               │
index2: ────────X───┼───@───────@───┼───X───────
                    │   │       │   │
index3: ────────X───┼───┼───@───┼───┼───X───────
                    │   │   │   │   │
chain0: ────────────X───@───┼───@───X───────────
                        │   │   │
chain1: ────────────────X───@───X───────────────

global phase:                               π


In [34]:
with_chain = reflect_by_chain.unitary(qubit_order=[*index, *chain])
assert np.allclose(with_chain[::4, ::4], reflect_matrix)
print("the AND chain's REFLECT is the walk's R")
print(f"the example's chain: {len(chain)} ANDs, and one more with the control bit")
print(
    f"carbon: {step.ancilla_qubits} ancilla qubits and the control bit, "
    f"{step.reflect_toffolis} ANDs = {weak.reflect_t} T per step"
)

the AND chain's REFLECT is the walk's R
the example's chain: 2 ANDs, and one more with the control bit
carbon: 43 ancilla qubits and the control bit, 42 ANDs = 168 T per step


## Why the walk encodes the energy: two mirrors make a rotation

Take an energy eigenstate, $H|\psi_E\rangle = E|\psi_E\rangle$, with $E$ measured from
the dropped constant, and follow $|0\rangle|\psi_E\rangle$ through one step.

1. The block encoding $B = \mathrm{PREPARE}^\dagger\,\mathrm{SELECT}\,\mathrm{PREPARE}$
   obeys the rule of "Block encoding (iv)":
   $B|0,\psi_E\rangle = \frac{E}{\lambda}|0,\psi_E\rangle + s\,|{\perp}\rangle$, with
   $s = \sqrt{1 - E^2/\lambda^2}$ and
   $|{\perp}\rangle$, the leftover, a state whose ancillas are away from $|0\rangle$.
2. $B$ is a mirror, $B^2 = I$, because every Pauli string squares to $I$ and PREPARE's
   inverse undoes PREPARE. So
   $B|{\perp}\rangle = s|0,\psi_E\rangle - \frac{E}{\lambda}|{\perp}\rangle$,
   and $B$ never leaves the plane of $|0,\psi_E\rangle$ and $|{\perp}\rangle$ (Low and
   Chuang, Quantum 3, 163 (2019)). $R$, the other mirror, keeps $|0,\psi_E\rangle$ and
   flips $|{\perp}\rangle$.
3. In that plane the step is
   $$W = RB = \begin{pmatrix} E/\lambda & s \\ -s & E/\lambda \end{pmatrix},$$
   a rotation by the angle $\theta_E$ with $\cos\theta_E = E/\lambda$. Two mirrors at an
   angle $\theta_E/2$ make a rotation by $\theta_E$, and the step's eigenvalues in the
   plane are $e^{\pm i\theta_E}$.
4. Steps add. $B$ alone would bounce back, since $B^2 = I$; with $R$ between, $k$ steps
   turn the state by $k\theta_E$ within the plane.

The energy is a rotation angle, exactly, with no approximation of the time evolution.
Roggero et al. 2019 write the step as
$Q = -i\,(2|G\rangle\langle G| - 1)\,\mathrm{SELECT}$, with
$|G\rangle = \mathrm{PREPARE}|0\rangle$ (their Eq. C5). That is
$-i\,\mathrm{PREPARE}\,W\,\mathrm{PREPARE}^\dagger$, so its eigenvalues are ours times
the fixed phase $-i$: they are $\mp e^{\pm i\arcsin(E/\lambda)}$ (their Eq. C6), since
$\arcsin x = \pi/2 - \arccos x$. Each of ours is turned by the same quarter turn, and
phase estimation reads either form.

The walk has no target. The state, an arrow in that plane, has nowhere to arrive: it
circles through the leftover and back, unmeasured, and phase estimation reads the angle.
More steps buy a finer reading of the angle. Amplitude amplification, in step 2 of the
recipe, uses the same geometry of two mirrors to reach a target, success ("Amplitude
amplification" below).

In the example $H$ is diagonal, so each of the 16 occupations of the site is an energy
eigenstate. The next cell builds each one's plane from the walk's own matrices, checks
that the step stays in it, and reads the angle from the step's eigenvalues there.

In [35]:
for occupation in range(16):
    entering = np.eye(256)[occupation]
    bounced = block_matrix @ entering
    across = bounced - bounced[occupation] * entering
    across /= np.linalg.norm(across)
    plane = np.column_stack([entering, across])
    in_plane = plane.conj().T @ walk_matrix @ plane
    assert np.allclose(walk_matrix @ plane, plane @ in_plane)
    turn = math.acos(site_matrix[occupation, occupation] / site_lambda)
    angles = np.sort(np.angle(np.linalg.eigvals(in_plane)))
    assert np.allclose(angles, [-turn, turn])
    if occupation == pair:
        pair_step = in_plane
pair_turn = math.acos(pair_energy / site_lambda)
print(
    "on all 16 occupations, one step stays in the plane and turns by arccos(E / lambda)"
)
print(f"two nucleons, p up and p down: E / lambda = {pair_energy / site_lambda:.4f}")
print(f"the step in their plane, a turn by {math.degrees(pair_turn):.2f} degrees:")
print(np.round(pair_step.real, 4))

on all 16 occupations, one step stays in the plane and turns by arccos(E / lambda)
two nucleons, p up and p down: E / lambda = -0.0701
the step in their plane, a turn by 94.02 degrees:
[[-0.0701  0.9975]
 [-0.9975 -0.0701]]


## Phase estimation (i): one extra qubit reads a phase

Suppose the system holds an eigenstate of the walk, $W|\phi\rangle = e^{i\theta}|\phi\rangle$.
A phase on the whole state cannot be seen. Put a control qubit in
$(|0\rangle + |1\rangle)/\sqrt{2}$ and apply $W$ only when it is $|1\rangle$: the system
is unchanged, and the control becomes $(|0\rangle + e^{i\theta}|1\rangle)/\sqrt{2}$. The
phase has moved onto the control (phase kickback). A Hadamard then gives
$\frac{1 + e^{i\theta}}{2}|0\rangle + \frac{1 - e^{i\theta}}{2}|1\rangle$, and a
measurement returns 0 with probability
$\left|\frac{1 + e^{i\theta}}{2}\right|^2 = \frac{1 + \cos\theta}{2}$: a little
information about $\theta$ per run.

The next cell builds that circuit for the example. Its wires, top to bottom: the
control qubit; the four index qubits, the ancilla register that names a term; and the
site's four modes, the system. Cirq applies the step as one gate, `W`, the matrix of
the walk circuit above, and numbers its wires `W[1]` to `W[8]`.

The state sent in is the one a run can prepare: the ancillas at $|0\rangle$ and the
site holding two protons, spin up and spin down, the occupation `1100` of "Block
encoding (iii)". $H$ is diagonal here, so that occupation is an energy eigenstate, with
its energy $E$ on the Hamiltonian's diagonal. It is not an eigenstate of the step. The
step's eigenstates in the pair's plane are the start state plus $i$ times the leftover
and the start state minus $i$ times the leftover, each over $\sqrt{2}$, with eigenvalues
$e^{\pm i\theta}$; the start state is half one and half the other, which the cell
checks. For this circuit the superposition costs nothing: each eigenstate sends the
control to 0 with probability $(1 + \cos\theta)/2$, the same number for $\theta$ and
$-\theta$, so the slide's "suppose an eigenstate" loses nothing here. $\theta$ is the
pair's walk angle from "Why the walk encodes the energy", $\arccos(E/\lambda)$; the cell
prints it with $E/\lambda$, and the control reads 0 a little less often than half the
time because the pair's energy sits a little below zero.

In [36]:
control = cirq.NamedQubit("control")
walk_gate = cirq.MatrixGate(walk_matrix, name="W")
one_bit = cirq.Circuit(
    cirq.H(control),
    walk_gate.on(*index, *site).controlled_by(control),
    cirq.H(control),
)
print(one_bit.to_text_diagram(qubit_order=[control, *index, *site]))
_, vectors = np.linalg.eig(pair_step)
shares = np.abs(vectors.conj().T @ np.array([1.0, 0.0])) ** 2
assert np.allclose(shares, [0.5, 0.5])
print(f"the pair state's shares of the plane's two eigenstates: {np.round(shares, 4)}")
entering = np.eye(512)[pair]
final = cirq.final_state_vector(
    one_bit,
    initial_state=entering.astype(np.complex128),
    qubit_order=[control, *index, *site],
    dtype=np.complex128,
)
reads_zero = np.sum(np.abs(final[:256]) ** 2)
assert math.isclose(reads_zero, (1 + math.cos(pair_turn)) / 2)
print(
    f"the pair's walk angle: {math.degrees(pair_turn):.2f} degrees, "
    f"E / lambda = {pair_energy / site_lambda:.4f}"
)
print(f"the control reads 0 with probability {reads_zero:.4f} = (1 + cos theta) / 2")

control: ───H───@──────H───
                │
index0: ────────W[1]───────
                │
index1: ────────W[2]───────
                │
index2: ────────W[3]───────
                │
index3: ────────W[4]───────
                │
p_up: ──────────W[5]───────
                │
p_down: ────────W[6]───────
                │
n_up: ──────────W[7]───────
                │
n_down: ────────W[8]───────
the pair state's shares of the plane's two eigenstates: [0.5 0.5]
the pair's walk angle: 94.02 degrees, E / lambda = -0.0701
the control reads 0 with probability 0.4650 = (1 + cos theta) / 2


## Phase estimation (ii): $W_q$ bits, and what each one costs

The energy register holds $W_q$ control qubits, bit $k$ driving $W^{2^k}$, so that bit
$k$ collects the phase $2^k\theta$. Together the register holds
$2^{-W_q/2}\sum_m e^{im\theta}|m\rangle$, and an inverse quantum Fourier transform turns
that into a peak at the $m$ with $2\pi m/2^{W_q}$ nearest $\theta$: the phase written in
binary, to $W_q$ bits. Near the middle of the spectrum, where $\theta \approx \pi/2$, a
step of $2\pi/2^{W_q}$ in the angle is a step of $2\pi\lambda/2^{W_q}$ in the energy:
that is the bin width.

The walk steps: $1 + 2 + 4 + \dots + 2^{W_q - 1} = 2^{W_q} - 1$ in the textbook version.
Babbush et al. run the walk forward on one branch and backward on the other, which
halves it to $2^{W_q - 1}$ (arXiv:1805.03662, Sec. II B). One more bit halves the bin and
doubles the walk: resolution is paid for in walk steps, and the steps are the cost.

The next cell builds the textbook version for the example with $W_q = 7$. A machine runs
$W^{2^k}$ as $2^k$ steps; Cirq takes the matrix power, a shortcut only a simulation has.
The cell after starts it from the pair state. There are two peaks, at $\theta$ and at
$2\pi - \theta$, one for each eigenvalue in the pair's plane, and both read back as the
same energy, $E = \lambda\cos(2\pi m/2^{W_q})$ plus the constant.

For carbon, `register_bits_for_sizing` sizes the register: the fewest bits with
$\lambda/2^{W_q}$ at most the case's sizing energy, 10 MeV. The sizing is a label, never
the resolution delivered. `weak.register`, the `EnergyRegister` that `energy_register`
computes, holds the register's bits, its walk steps and its bin width at mid-band.

In [37]:
energy_bits = [cirq.NamedQubit(f"bit{k}") for k in range(7)]
qpe = cirq.Circuit(cirq.H(bit) for bit in energy_bits)
for k, bit in enumerate(energy_bits):
    power = cirq.MatrixGate(np.linalg.matrix_power(walk_matrix, 2**k), name=f"W^{2**k}")
    qpe.append(power.on(*index, *site).controlled_by(bit))
qpe.append(cirq.qft(*reversed(energy_bits), inverse=True))
register_order = [*reversed(energy_bits), *index, *site]
print(qpe.to_text_diagram(qubit_order=register_order))

bit6: ─────H───────────────────────────────────────────────────────────@─────────qft^-1───
                                                                       │         │
bit5: ─────H─────────────────────────────────────────────────@─────────┼─────────#2───────
                                                             │         │         │
bit4: ─────H───────────────────────────────────────@─────────┼─────────┼─────────#3───────
                                                   │         │         │         │
bit3: ─────H──────────────────────────────@────────┼─────────┼─────────┼─────────#4───────
                                          │        │         │         │         │
bit2: ─────H─────────────────────@────────┼────────┼─────────┼─────────┼─────────#5───────
                                 │        │        │         │         │         │
bit1: ─────H────────────@────────┼────────┼────────┼─────────┼─────────┼─────────#6───────
                        │        │     

In [38]:
entering = np.eye(2**15)[pair]
final = cirq.final_state_vector(
    qpe,
    initial_state=entering.astype(np.complex128),
    qubit_order=register_order,
    dtype=np.complex128,
)
readings = np.sum(np.abs(final.reshape(2**7, 256)) ** 2, axis=1)
bin_width = 2 * math.pi * site_lambda / 2**7
for m in np.argsort(readings)[::-1][:4]:
    energy = site_lambda * math.cos(2 * math.pi * m / 2**7) + coefficients.identity_mev
    print(f"m = {m:3d}: probability {readings[m]:.3f}, E = {energy:7.2f} MeV")
best = int(np.argmax(readings))
best_energy = (
    site_lambda * math.cos(2 * math.pi * best / 2**7) + coefficients.identity_mev
)
assert abs(best_energy - C0) < bin_width
print(
    f"C0 = {C0} MeV; a bin near the middle of the spectrum is {bin_width:.1f} MeV wide"
)

m =  33: probability 0.262, E =  -93.79 MeV
m =  95: probability 0.262, E =  -93.79 MeV
m =  94: probability 0.148, E = -104.13 MeV
m =  34: probability 0.148, E = -104.13 MeV
C0 = -98.23 MeV; a bin near the middle of the spectrum is 10.4 MeV wide


In [39]:
register = weak.register
sized_bits = register_bits_for_sizing(lattice, carbon.sizing_mev)
print(f"carbon: the sizing rule at {carbon.sizing_mev:g} MeV gives {sized_bits} bits")
print(f"energy register: {register.bits} bits")
print(f"walk steps, forward and backward: {register.walk_steps:,}")
print(f"bin width at mid-band: {register.bin_width_mev:.1f} MeV")

carbon: the sizing rule at 10 MeV gives 12 bits
energy register: 12 bits
walk steps, forward and backward: 2,048
bin width at mid-band: 33.2 MeV


## Phase estimation (iii): what one run returns

A superposition in, one energy out. Once the current is applied, the system is a
superposition of energy eigenstates with weights $|\langle f|J|\Psi_0\rangle|^2$, and
phase estimation returns one eigenstate's energy bin, picked in proportion to that
weight, with probability $|\langle f|J|\Psi_0\rangle|^2/\eta^2$: one sample of the
response. Each successful run returns one bin, and over many runs the counts are the
response in those bins. The next cell shows it on the example, starting from an equal
superposition of the pair state and a triple, $|1110\rangle$.

Each energy is sharp, but its reading has a width, as the readings above show: the
register sees only $2^{W_q}$ steps of the walk, a finite time, which blurs an energy by
about one bin, so an energy between two of the register's angles is read into both, with
tails farther out. Starting the register in a window state, in place of an equal
superposition, cuts the tails (Babbush et al., Eq. 17). The error carbon's circuit
delivers, the window's lineshape and the circuit's error budget added in quadrature, is
$\sqrt{2}\,\pi\lambda/2^{W_q}$ RMS (ibid., Eq. 24), `EnergyRegister.rms_error_mev`;
that, and not the bin width, is the resolution the lecture quotes.

In [40]:
triple = int("1110", 2)
entering = (np.eye(2**15)[pair] + np.eye(2**15)[triple]) / math.sqrt(2)
final = cirq.final_state_vector(
    qpe,
    initial_state=entering.astype(np.complex128),
    qubit_order=register_order,
    dtype=np.complex128,
)
readings = np.sum(np.abs(final.reshape(2**7, 256)) ** 2, axis=1)
angles = 2 * np.pi * np.arange(2**7) / 2**7
energies_read = site_lambda * np.cos(angles) + coefficients.identity_mev
for name, energy in [("the pair", C0), ("the triple", 3 * C0 + D0)]:
    near = np.abs(energies_read - energy) < 2 * bin_width
    read_near = readings[near].sum()
    print(f"{name}, E = {energy:.2f} MeV: read within two bins of it, {read_near:.3f}")
    assert read_near > 0.4
print(f"carbon's delivered error: {register.rms_error_mev:.1f} MeV RMS")

the pair, E = -98.23 MeV: read within two bins of it, 0.456
the triple, E = -166.85 MeV: read within two bins of it, 0.476
carbon's delivered error: 23.5 MeV RMS


What does the machine hold after the reading? Textbook phase estimation runs on
$e^{-iHt}$, which acts on the system alone, and leaves the system in the eigenstate
whose energy it read. Ours runs on the walk, which acts on the system and the ancillas
together, so it leaves an eigenstate of the walk,
$(|0\rangle|\psi_E\rangle \pm i|{\perp}\rangle)/\sqrt{2}$: the energy eigenstate and its
leftover in equal parts, one sign for each of the two peaks of "Phase estimation (ii)".

Step 3 of the recipe never uses that state: it reads an energy and the run ends. A
preparation that projects by phase estimation would, since the state a reading leaves is
the state it keeps ("Why carbon's preparation is still unpriced: our view" below). The
repair is cheap, and it takes one particular measurement: a test of whether the ancillas
are all zero that learns nothing else about them. An AND chain like REFLECT's writes the
answer into one fresh qubit, and only that qubit is measured. Half the time it says yes,
and the system holds $|\psi_E\rangle$. Otherwise the machine holds the whole leftover.
After one more use of $B$, another test says yes with probability $1 - E^2/\lambda^2$,
by the rule for $B|{\perp}\rangle$ in "Why the walk encodes the energy", with $E$
measured from the dropped constant as there; a no leaves the leftover again, so the step
repeats. Reading the ancillas one by one would spoil it: when they are not all zero, the
reading shows which value they hold, and that keeps only one piece of the leftover,
while the rule is about the whole of it.

Roggero et al. 2019 name this step, a rotation from the walk's eigenvectors to those of
$H$ (their p. 6), and point to Poulin et al., Phys. Rev. Lett. 121, 010501 (2018), who
make the same test; after a no they run the phase estimation again, where this section
uses $B$ once. The next cell does it to the run above, at the likeliest reading near
each of the two energies. For the test's no, it removes the part of the state with the
ancillas all zero and keeps the rest whole.

In [41]:
left_behind = final.reshape(2**7, 16, 16)
for line_name, line_state, line_energy in [
    ("the pair", pair, C0),
    ("the triple", triple, 3 * C0 + D0),
]:
    line_bins = np.abs(energies_read - line_energy) < 2 * bin_width
    line_reading = int(np.argmax(np.where(line_bins, readings, 0)))
    state_left = left_behind[line_reading]
    zero_chance = np.sum(np.abs(state_left[0]) ** 2) / readings[line_reading]
    state_share = np.abs(state_left[0, line_state]) ** 2 / np.sum(
        np.abs(state_left[0]) ** 2
    )
    leftover_alone = state_left.copy()
    leftover_alone[0] = 0
    leftover_alone = leftover_alone.reshape(256) / np.linalg.norm(leftover_alone)
    second_look = (block_matrix @ leftover_alone).reshape(16, 16)
    rescue_chance = np.sum(np.abs(second_look[0]) ** 2)
    rescue_share = np.abs(second_look[0, line_state]) ** 2 / rescue_chance
    rescue_expected = 1 - ((line_energy - coefficients.identity_mev) / site_lambda) ** 2
    assert abs(zero_chance - 0.5) < 0.02
    assert state_share > 0.99
    assert abs(rescue_chance - rescue_expected) < 0.01
    assert rescue_share > 0.99
    print(f"after a reading at the energy of {line_name}:")
    print(f"  the ancillas read all zero with probability {zero_chance:.3f}")
    print(f"  the system then holds {line_name} with probability {state_share:.3f}")
    print(
        f"  otherwise, one more B and all zero with probability {rescue_chance:.3f}; "
        f"1 - (E / lambda)^2 = {rescue_expected:.3f}"
    )

after a reading at the energy of the pair:
  the ancillas read all zero with probability 0.506
  the system then holds the pair with probability 0.997
  otherwise, one more B and all zero with probability 0.995; 1 - (E / lambda)^2 = 0.995
after a reading at the energy of the triple:
  the ancillas read all zero with probability 0.507
  the system then holds the triple with probability 0.998
  otherwise, one more B and all zero with probability 0.845; 1 - (E / lambda)^2 = 0.844


# The cost, piece by piece

The second part prices carbon's circuit, piece by piece, from the module's counts.

## One walk step, priced

What does one step of the quantum walk cost? The sections above opened its four boxes;
this one adds them up for carbon. A Toffoli, built as an AND, costs 4 $T$, and a
rotation costs what the error budget sets, `weak.synthesis.t_per_rotation` (see the
rotations below). The `OneCircuit` properties `prepare_pair_t`, `select_t`, `reflect_t`
and `walk_step_t` price the pieces. SELECT and REFLECT are priced controlled on every
step, and the reflection covers every ancilla PREPARE touches; the docstring of
`walk_step` sizes what a leaner construction would save, a small share of a step. No
circuit is computed for carbon: each line is a counting argument on a published
construction, and that is what a resource estimate is.

In [42]:
print(f"price per rotation: {weak.synthesis.t_per_rotation} T")
print(f"keep number: {step.keep_bits} bits")
print(
    f"PREPARE and its inverse: 2 x {step.prepare_toffolis} Toffolis and "
    f"2 x {step.prepare_rotations} rotations = {weak.prepare_pair_t:,} T"
)
print(
    f"SELECT: {step.select_toffolis:,} Toffolis = {weak.select_t:,} T, "
    f"{100 * weak.select_share:.0f}% of the step"
)
print(
    f"REFLECT: {step.ancilla_qubits} ancilla qubits, "
    f"{step.reflect_toffolis:,} ANDs = {weak.reflect_t:,} T"
)
print(f"one step: {weak.walk_step_t:,} T")

price per rotation: 34 T
keep number: 12 bits
PREPARE and its inverse: 2 x 65 Toffolis and 2 x 2 rotations = 656 T
SELECT: 2,431 Toffolis = 9,724 T, 92% of the step
REFLECT: 43 ancilla qubits, 42 ANDs = 168 T
one step: 10,548 T


## The walk and the readout, priced

What does one energy measurement cost? It has two parts: the walk, and the readout
around it.

The walk: phase estimation runs $2^{W_q - 1}$ steps of the walk in all ("Phase
estimation (ii)"), and the section above added up the $T$ gates of one step. `walk_t`
multiplies the two: the number of steps times the $T$ count of one step.

The readout: before the walk, the energy register is put into its window state, the
starting state from "Phase estimation (iii)" that cuts the tails of the lineshape; after
the walk, an inverse quantum Fourier transform turns the phases the register has
collected into the bits of the energy. Both are built from rotations, and each rotation
costs $T$ gates (the rotation sections below say how many). `readout_rotations` counts
them with two upper bounds: $2^{W_q}$ rotations for the window state, one per amplitude
it sets, and $W_q (W_q - 1)$ for the transform, twice the number of its controlled
phases. Babbush et al. build the window state with a gate count that grows only about
linearly in $W_q$ (their Fig. 2 and Eqs. 17 to 20), but at $2^{W_q}$ rotations it is
already a small part of the measurement, so the simple count stays. `readout_t` prices
the rotations, and `energy_measurement_t` adds the walk and the readout.

The cells give each count as a share of the 2033 budget, `HARD_OPERATIONS_2033`, $10^9$
hard operations in one circuit: the scale for the early-to-mid 2030s in the U.S.
Department of Energy's request for information RFI 892431-26-RFI-0001. Each Toffoli
counts as 4 $T$ against that budget. The budget's wording would also let a Toffoli count
as one hard operation, so counting four is the cautious choice: a circuit that fits this
way fits either way. If the ground state were free, carbon would fit the budget with
room to spare; it is not free, and it is loaded tens of times per circuit.

In [43]:
budget = HARD_OPERATIONS_2033
readout = readout_rotations(register.bits)
measurement = weak.energy_measurement_t
print(f"the walk: {weak.walk_t:,} T, {100 * weak.walk_t / budget:.3g}% of the budget")
print(
    f"the readout: at most {readout.fourier} rotations in the inverse Fourier "
    f"transform and {readout.window:,} in the window state"
)
print(f"  = {weak.readout_t:,} T, {100 * weak.readout_t / budget:.3g}% of the budget")
print(
    f"the energy measurement: {measurement:,} T ({measurement:.2e}), "
    f"{100 * measurement / budget:.3g}% of the budget"
)

the walk: 21,602,304 T, 2.16% of the budget
the readout: at most 132 rotations in the inverse Fourier transform and 4,096 in the window state
  = 143,752 T, 0.0144% of the budget
the energy measurement: 21,746,056 T (2.17e+07), 2.17% of the budget


The size of the lattice drives this cost twice. $\lambda$ sets the energy register's
scale: $W_q$ bits read energies in bins of $2\pi\lambda/2^{W_q}$, as "Phase estimation
(ii)" showed, so a larger $\lambda$ takes more bits, and more walk steps, for the same
bin. $\Gamma$ sets the cost of each step, through SELECT. Both grow with the number of
sites. The next cell prices carbon's energy measurement on the $4^3$ lattice and on the
next, $5^3$, with the register sized as before: $\lambda$ nearly doubles, which takes
one more bit and twice the steps, and $\Gamma$ nearly doubles the cost of each step. The
energy measurement is the same at both ends of the strength band, so the cell keeps the
weak end.

In [44]:
circuits = []
for box_side in (4, 5):
    case = LectureCase(
        nucleus=carbon.nucleus, box_side=box_side, sizing_mev=carbon.sizing_mev
    )
    circuit, _ = price_lecture_case(case)
    circuits.append(circuit)
    print(
        f"carbon on {box_side}^3: lambda = {circuit.lattice.one_norm_mev:,.0f} MeV, "
        f"Gamma = {circuit.lattice.term_count:,}, {circuit.register.bits} bits, "
        f"{circuit.register.walk_steps:,} steps of {circuit.walk_step_t:,} T"
    )
small_box, large_box = circuits
lambda_ratio = large_box.lattice.one_norm_mev / small_box.lattice.one_norm_mev
gamma_ratio = large_box.lattice.term_count / small_box.lattice.term_count
steps_ratio = large_box.register.walk_steps / small_box.register.walk_steps
step_t_ratio = large_box.walk_step_t / small_box.walk_step_t
measurement_ratio = large_box.energy_measurement_t / small_box.energy_measurement_t
print(
    f"from 4^3 to 5^3: lambda x {lambda_ratio:.2f}, steps x {steps_ratio:.0f}, "
    f"Gamma x {gamma_ratio:.2f}, T per step x {step_t_ratio:.2f}"
)
print(
    f"the energy measurement: {small_box.energy_measurement_t:.3g} T to "
    f"{large_box.energy_measurement_t:.3g} T, x {measurement_ratio:.1f}"
)

carbon on 4^3: lambda = 21,646 MeV, Gamma = 2,432, 12 bits, 2,048 steps of 10,548 T
carbon on 5^3: lambda = 42,278 MeV, Gamma = 4,750, 13 bits, 4,096 steps of 20,104 T
from 4^3 to 5^3: lambda x 1.95, steps x 2, Gamma x 1.95, T per step x 1.91
the energy measurement: 2.17e+07 T to 8.26e+07 T, x 3.8


## Rotations (i): every angle is built from $T$ gates

Where are the rotations? The prepare's angles, four per walk step, the inverse Fourier
transform and the window state: each an $R_z(\theta)$ at an arbitrary angle. The ground
state's preparation, unpriced, adds its own. A fault-tolerant machine has only Clifford
gates and $T$, and approximates each $R_z(\theta)$ to a precision $\epsilon$ by a
sequence of them, with a $T$ count that grows like $\log(1/\epsilon)$.

Why measure anything? A fixed sequence of Clifford gates and $T$ reaches only a finite
set of unitaries, spread over every axis of rotation, so landing within $\epsilon$ of
the $R_z(\theta)$ you want costs about $3\log_2(1/\epsilon)$ $T$ (Ross and Selinger,
arXiv:1403.2975). Repeat-until-success synthesis (Bocharov, Roetteler and Svore 2015,
arXiv:1404.5320) makes the same move as the $T$ gate by teleportation: run a short
two-qubit sequence on the target and an ancilla qubit, then measure the ancilla. On one
outcome the target has received a $z$-rotation, from a far denser set of angles for the
same $T$ count; on the other it has received a known error that a Clifford gate undoes,
and the circuit tries again. The module prices each rotation this way, the cheaper of
the two methods the notebook prices: about half the $T$ of Ross and Selinger's sequence
at this circuit's precision (the last cell of the next section). Counting every try, it
needs on average, in the form of Watson et al. (arXiv:2312.05344, Eq. 80),

$$N_T(\epsilon) = \left\lceil 1.15 \log_2(2/\epsilon) + 9.2 \right\rceil$$

$T$ gates, `repeat_until_success_t_count`. The next question is what $\epsilon$ to ask
for.

In [45]:
synthesis = weak.synthesis
count = synthesis.rotation_count
print(f"rotations and rounded angles in what is priced: {count:,}")
for precision in (1e-3, 1e-6, 1e-9):
    print(f"precision {precision:.0e}: {repeat_until_success_t_count(precision)} T")

rotations and rounded angles in what is priced: 12,420
precision 1e-03: 22 T
precision 1e-06: 34 T
precision 1e-09: 45 T


## Rotations (ii): the error budget sets $34$ $T$ per rotation

Why does a rotation cost tens of $T$? The circuit's approximations share one error
budget, `SYNTHESIS_ERROR_BUDGET`, 0.01 per circuit. That budget is a bias we choose to
tolerate, the circuit's unitary being slightly off in the same way in every run; it is
not the machine's logical error rate, which is a different kind of error with its own
budget (see the error budget below). The approximations' errors are coherent, each off
by a small amount in a fixed direction, and add linearly, so `synthesis_budget` shares
the budget evenly over the $N$ rotations and rounded angles priced, $\epsilon = 0.01/N$
each. `weak.synthesis` is the `Synthesis` it returns. A preparation's own rotations are
in $N$ only where the preparation is priced, and carbon's is not (see the ground state
below). The budget is a weak lever, since the count grows only with $\log(1/\epsilon)$:
a decade either way moves the price by a few $T$, and rotations are a small share of
what is priced, which the cell prints.

The second cell gives the randomized alternative. Mixing approximations of each rotation
at random keeps each within $5\epsilon^2$ (Campbell 2017, arXiv:1612.02689, Thm. 2),
which allows a coarser precision. That theorem covers a deterministic synthesis such as
Ross and Selinger's (arXiv:1403.2975), and not repeat-until-success;
`ross_selinger_t_estimate` prices their method, $3 \log_2 (1/\epsilon)$ $T$. The loose
tolerance and the cheap method do not combine.

In [46]:
looser = synthesis_budget(count, error_budget=10 * SYNTHESIS_ERROR_BUDGET)
tighter = synthesis_budget(count, error_budget=SYNTHESIS_ERROR_BUDGET / 10)
print(f"precision per rotation: {synthesis.precision:.2e}")
print(f"price per rotation: {synthesis.t_per_rotation} T")
print(
    f"a budget 10 times looser: {looser.t_per_rotation} T; "
    f"10 times tighter: {tighter.t_per_rotation} T"
)
print(f"rotations' share of everything priced: {100 * weak.rotation_share:.2g}%")

precision per rotation: 8.05e-07
price per rotation: 34 T
a budget 10 times looser: 30 T; 10 times tighter: 38 T
rotations' share of everything priced: 1.9%


In [47]:
randomized = randomized_synthesis_precision(count)
print(f"randomized compiling: precision {randomized:.2e} per rotation")
print(f"Ross-Selinger at that precision: {ross_selinger_t_estimate(randomized):.0f} T")
print(
    "Ross-Selinger at the coherent precision: "
    f"{ross_selinger_t_estimate(synthesis.precision):.0f} T"
)

randomized compiling: precision 4.01e-04 per rotation
Ross-Selinger at that precision: 34 T
Ross-Selinger at the coherent precision: 61 T


## Loading a known state (i): a table of amplitudes onto qubits

Suppose a classical computer holds the ground state as a table: $D$ configurations
$\nu_i$, bit strings with one bit per mode, and their amplitudes $\alpha_i$, with $D$
far smaller than the $2^{\text{modes}}$ strings the register could hold. Fomichev et al.
load $\sum_i \alpha_i|\nu_i\rangle$ onto the system register in six steps
(arXiv:2310.18410, Sec. III). Three registers take part. The *system register* is the
one the whole lecture is about, one qubit per mode: in the running example, the site's
four modes `p_up`, `p_down`, `n_up`, `n_down`, the bottom four wires of the phase
estimation circuits. The four index qubits of those circuits belong to the block
encoding, which names a term with them; the load does not touch them. The other two
registers are ancilla registers the load brings, both starting at $|0\rangle$ and both
returned to $|0\rangle$ by the end, as the circuit words require: an *enumeration
register* of $L = \lceil\log_2 D\rceil$ qubits, which holds a row number of the table,
and an *identification register* of $2L - 1$ qubits, which holds a short identifier of
a configuration, read off the configuration's own bits; the next section says what it
is for and why it has $2L - 1$ bits. Both count among the load's ancillas in "The qubit
count".

- **Step 1, the amplitudes, on a small register.** Number the table's rows
  $i = 0, 1, \ldots, D - 1$ and make $\sum_i \alpha_i|i\rangle$ on the enumeration
  register, where $|i\rangle$ is the row number written in binary on its $L$ qubits.
  This is a dense state, an amplitude on every row, but on few qubits, and rotations
  make it.
- **Step 2, the configurations, by a table lookup.** For each row $i$, write that row's
  configuration $\nu_i$ into the system register when the enumeration register reads
  $i$: X gates on the system qubits that are 1 in $\nu_i$, controlled on the row
  number, a tree of ANDs like SELECT's at $D$ Toffolis. The state is now
  $\sum_i \alpha_i|\nu_i\rangle|i\rangle$: the right configurations with the right
  amplitudes, each still tagged by its row number.
- **Step 3, the first bit of the identifier.** Choose a set of system bits on the
  classical computer and compute their parity, the sum of those bits mod 2, which is 1
  when an odd number of them are 1. A parity is a chain of CNOTs: one CNOT from each
  chosen system qubit into the first qubit of the identification register. No Toffoli.
- **Step 4, the other $2L - 2$ bits of the identifier,** each the parity of its own
  chosen set of system bits, by CNOTs into the remaining qubits of the identification
  register in the same way; the identifier then has $2L - 1$ bits in all. The sets are
  chosen with the table in hand, so that no two of the $D$ configurations get the same
  identifier; the paper's Lemma III.1 shows that $2L - 1$ parities always suffice, and
  its App. C gives the search that finds them. The identifier is a fingerprint of the
  configuration: $2L - 1$ bits in place of one bit per mode.
- **Step 5, the row number erased.** For each row $i$, flip the qubits of the
  enumeration register that are 1 in the binary $i$ when the identification register
  reads row $i$'s identifier: one operation controlled on all $2L - 1$ identifier bits
  per row, an AND chain of $2L - 2$ ANDs. The enumeration register is back at
  $|0\rangle$ on every branch. This is the step that costs most.
- **Step 6, the identification register uncomputed.** Run the CNOTs of steps 3 and 4
  again: a CNOT is its own inverse, so the identifier is erased and the identification
  register is back at $|0\rangle$. The system holds $\sum_i \alpha_i|\nu_i\rangle$
  alone, with both ancilla registers clean.

To see the six steps in a small case, the next cells load the state that "Phase
estimation (iii)" sent in as a given vector: the pair occupation `1100` and the triple
`1110` with equal amplitudes, $(|1100\rangle + |1110\rangle)/\sqrt{2}$. As a table it
has $D = 2$ rows, so the enumeration register is $L = 1$ qubit, named `row`, and the
identification register $2L - 1 = 1$ qubit, named `identifier`. The next cell runs
steps 1 and 2: one rotation puts equal amplitudes on $|0\rangle$ and $|1\rangle$ of
`row`, and the lookup writes `1100` beside $|0\rangle$ and `1110` beside $|1\rangle$.
It then describes the system on its own, averaging over whatever the ancillas hold (the
partial trace), and prints the weights of the two configurations, the cross term
between them and the purity $\mathrm{tr}\,\rho^2$, which is 1 for a pure state and
smaller for a mixture.

In [48]:
row_qubit = cirq.NamedQubit("row")
identifier_qubit = cirq.NamedQubit("identifier")
known_table = {0: "1100", 1: "1110"}
known_amplitudes = np.array([1.0, 1.0]) / math.sqrt(2)
lookup = cirq.Circuit(cirq.ry(2 * math.acos(known_amplitudes[0])).on(row_qubit))
for number, configuration in known_table.items():
    for bit, qubit in zip(configuration, site, strict=True):
        if bit == "1":
            lookup.append(
                cirq.X(qubit).controlled_by(row_qubit, control_values=[number])
            )
print(lookup.to_text_diagram(qubit_order=[row_qubit, *site]))
after_lookup = cirq.final_state_vector(
    lookup, qubit_order=[*site, row_qubit], dtype=np.complex128
)
tagged = np.zeros(32)
tagged[pair * 2 + 0] = tagged[triple * 2 + 1] = 1 / math.sqrt(2)
assert np.allclose(after_lookup, tagged)
system_alone = after_lookup.reshape(16, 2)
rho = system_alone @ system_alone.conj().T
print("after step 2, the system alone:")
print(
    f"  weight on 1100: {rho[pair, pair].real:.2f}, "
    f"on 1110: {rho[triple, triple].real:.2f}"
)
print(f"  the cross term between them: {rho[pair, triple].real:.2f}")
print(f"  purity tr(rho^2): {np.trace(rho @ rho).real:.2f}, a mixture, not the state")

row: ──────Ry(0.5π)───(0)───(0)───@───@───@───
                      │     │     │   │   │
p_up: ────────────────X─────┼─────X───┼───┼───
                            │         │   │
p_down: ────────────────────X─────────X───┼───
                                          │
n_up: ────────────────────────────────────X───

n_down: ──────────────────────────────────────
after step 2, the system alone:
  weight on 1100: 0.50, on 1110: 0.50
  the cross term between them: 0.00
  purity tr(rho^2): 0.50, a mixture, not the state


## Loading a known state (ii): erasing the row number is the load

After step 2 the system does not yet hold the state. It holds
$\sum_i \alpha_i|\nu_i\rangle|i\rangle$, each configuration tied to its row number: an
ancilla left changed, which the circuit words said spoils the state. Anything acting on
the system alone sees a *mixture*, as if one configuration had been drawn at random with
probability $|\alpha_i|^2$: its density matrix is
$\sum_i |\alpha_i|^2 |\nu_i\rangle\langle\nu_i|$, with no cross terms. The interference
between configurations that makes the sum an eigenstate is gone, and phase estimation
would read a spread of energies in place of $E_0$. Measuring the row number does not
help: the measurement picks one row and leaves the system in that single configuration.

Erasing it means a reverse lookup. Running step 2 backwards does not do it: a controlled
X is its own inverse, so step 2 run again erases the configurations, controlled on the
row number, and leaves $\sum_i \alpha_i|0\rangle|i\rangle$, step 1's state with the
system empty. What is wanted is the other direction: for each row $i$, flip the
enumeration register from $|i\rangle$ back to $|0\rangle$ when the system register reads
$\nu_i$. That works as it stands, and nothing is measured along the way, since a
controlled operation acts on every branch of the superposition at once. It is only
expensive: recognizing a whole configuration is an AND chain over every system qubit,
and for helium on the $3^3$ lattice that is $108$ system qubits, four modes on each of
$27$ sites, so $107$ ANDs per row and $107D$ Toffolis in all.

The identifier is the economy (steps 3 and 4). Only $D$ configurations are in play, so
$2L - 1$ parities of well-chosen bits already tell them apart (the paper's Lemma III.1),
and a parity costs CNOTs, no Toffoli. Step 5 then recognizes identifiers of $2L - 1$
bits in place of whole configurations: $2L - 2$ ANDs per row, $(2L - 2)D$ in all, which
for helium is $28$ ANDs per row in place of $107$. The Toffolis of the erasure stay, and
they remain the largest piece of the load; the identifier shortens the string they have
to recognize. Step 6 repeats the CNOTs, which clears the identifiers.

Why two registers, then? Each does what the other cannot. The row number is a dense
address: $L$ bits, most of whose values are rows, so step 1 can put the amplitudes on it
and step 2 can look the configurations up from it with a plain tree. But nothing in the
system register says which row a configuration came from. The table is arbitrary, so
recovering the row from the configuration is a recognition, ANDs per row, and the row
number cannot be erased with CNOTs. The identifier is read off the configuration itself,
and linearly: each bit is a parity of system bits, so CNOTs compute it and the same
CNOTs erase it. But its values are sparse, $D$ of the $2^{2L-1}$ it could take, so
putting the amplitudes on it in step 1 would be the sparse-state problem we started
with. Step 5 is the one place the two meet: the row number erased, controlled on the
identifier. Why $2L - 1$ bits, and not the row number's $L$? A parity is a linear
function of the bits, a sum mod 2, so two configurations get the same identifier exactly
when their difference, the bitwise XOR, has an even number of ones in every chosen set.
$L$ bits are enough to label $D$ rows, but no choice of $L$ parities need keep a given
table's configurations apart. Count the pairs: there are $D(D - 1)/2$ of them, and $m$
parity sets drawn at random all miss a given pair's difference with probability
$2^{-m}$, so the expected number of pairs that collide is $D(D - 1)/2^{m + 1}$, which
drops below 1 at $m = 2L - 1$, since $D \le 2^L$. Some choice of $2L - 1$ sets therefore
separates every pair; the paper's App. C makes the count constructive, a search that
finds the sets. This is the birthday paradox read the other way: about $2\log_2 D$ bits
keep $D$ labels from colliding. The paper conjectures that $2L - 1$ cannot be beaten for
every table; a particular table may allow fewer. In the example, $D = 2$ and $L = 1$, so
the identifier is one bit either way.

Steps 3 and 4 leave the system's bits as they were, since a CNOT never changes its
control, but they do change the state: the identification register now holds a value
that differs from branch to branch, a second tag in place of the first. After step 5 the
row number is gone and the system is still a mixture, tied now to the identifier. The
next cell shows it on the example: with the identifier written and the row number
erased, the cross term is $0$ and the purity $0.5$, as after step 2; once step 6 has
cleared the identifier, the cross term is $0.5$ and the purity $1$. Step 6 is not
tidying: without it the load fails as it did after step 2.

The price (ibid., Eq. 17): the amplitudes $2^{L+1}$ Toffolis (under $4D$, since
$2^L < 2D$), the lookup $D$, the erasure $(2L - 2)D$, the CNOT steps nothing: fewer than
$(2L + 3)D$ Toffolis in all, most of them the erasure. Step 1's $L$ rotations are not
synthesized as the walk's are: the paper adds each angle, held as a binary number of
$c$ bits, into a register prepared once for the purpose, at $c - 3$ Toffolis per angle
(their App. D), a small addition the library counts. In the example the identifier is a
single bit, the parity of the one mode where the two rows differ, `n_up`, so steps 3, 5
and 6 are one CNOT each and nothing in the load is a Toffoli: the Toffolis come with
the identifier's length, since a flip conditioned on $2L - 1$ bits needs an AND chain.
The next cell runs steps 3 to 6 on the example and checks the state; the cell after
sets helium's table, the one the lecture prices, against the bound and shows where its
Toffolis fall.

In [49]:
identify = cirq.Circuit(cirq.CNOT(site[2], identifier_qubit))
erase_row = cirq.Circuit(
    cirq.X(row_qubit).controlled_by(identifier_qubit, control_values=[1])
)
six_steps = lookup + identify + erase_row + identify
print(six_steps.to_text_diagram(qubit_order=[row_qubit, identifier_qubit, *site]))
after_step_5 = cirq.final_state_vector(
    lookup + identify + erase_row,
    qubit_order=[*site, row_qubit, identifier_qubit],
    dtype=np.complex128,
).reshape(16, 4)
assert np.allclose(after_step_5[:, 2:], 0)
rho = after_step_5 @ after_step_5.conj().T
print("after step 5, the row number erased, the identifier still written:")
print(f"  the cross term between 1100 and 1110: {rho[pair, triple].real:.2f}")
print(
    f"  purity tr(rho^2): {np.trace(rho @ rho).real:.2f}, "
    "a mixture still, tied to the identifier"
)
loaded = cirq.final_state_vector(
    six_steps, qubit_order=[*site, row_qubit, identifier_qubit], dtype=np.complex128
)
pair_and_triple = (np.eye(16)[pair] + np.eye(16)[triple]) / math.sqrt(2)
assert np.allclose(loaded.reshape(16, 4)[:, 0], pair_and_triple)
assert np.allclose(loaded.reshape(16, 4)[:, 1:], 0)
rho = loaded.reshape(16, 4) @ loaded.reshape(16, 4).conj().T
print("after step 6, the system alone:")
print(f"  the cross term between 1100 and 1110: {rho[pair, triple].real:.2f}")
print(f"  purity tr(rho^2): {np.trace(rho @ rho).real:.2f}, the state, ancillas at 0")
assert all(len(operation.qubits) <= 2 for operation in six_steps.all_operations())
print("every gate of this load acts on one or two qubits: no Toffoli at this size")

row: ──────────Ry(0.5π)───(0)───(0)───@───@───@───────X───────
                          │     │     │   │   │       │
identifier: ──────────────┼─────┼─────┼───┼───┼───X───@───X───
                          │     │     │   │   │   │       │
p_up: ────────────────────X─────┼─────X───┼───┼───┼───────┼───
                                │         │   │   │       │
p_down: ────────────────────────X─────────X───┼───┼───────┼───
                                              │   │       │
n_up: ────────────────────────────────────────X───@───────@───

n_down: ──────────────────────────────────────────────────────
after step 5, the row number erased, the identifier still written:
  the cross term between 1100 and 1110: 0.00
  purity tr(rho^2): 0.50, a mixture still, tied to the identifier
after step 6, the system alone:
  the cross term between 1100 and 1110: 0.50
  purity tr(rho^2): 1.00, the state, ancillas at 0
every gate of this load acts on one or two qubits: no Toffoli at this size


In [50]:
helium = LECTURE_CASES["helium"]
helium_weak, helium_strong = price_lecture_case(helium)
helium_load = helium_weak.load
enumeration = enumeration_register_qubits(helium_load.amplitudes)
load_pieces = {
    "the amplitudes, 2^(L + 1)": 2 ** (enumeration + 1),
    "the lookup, D": helium_load.amplitudes,
    "the erasure, (2L - 2) D": (2 * enumeration - 2) * helium_load.amplitudes,
}
load_rotations = helium_load.load_toffolis - sum(load_pieces.values())
assert load_rotations == (helium_load.gradient_qubits - 3) * enumeration
load_bound = (2 * enumeration + 3) * helium_load.amplitudes
assert helium_load.load_toffolis < load_bound
print(
    f"helium's table: D = {helium_load.amplitudes:,} configurations, L = {enumeration}"
)
for name, toffolis in load_pieces.items():
    print(f"  {name}: {toffolis:,} Toffolis")
print(
    f"  step 1's {enumeration} rotations, (c - 3) L with c = "
    f"{helium_load.gradient_qubits} bits per angle: {load_rotations:,} Toffolis"
)
print(
    f"the library counts {helium_load.load_toffolis:,} Toffolis; "
    f"the bound (2L + 3) D = {load_bound:,}"
)
helium_system_qubits = helium_weak.qubits.system
print(
    f"recognizing whole configurations instead: ({helium_system_qubits} - 1) D = "
    f"{(helium_system_qubits - 1) * helium_load.amplitudes:,} Toffolis"
)

helium's table: D = 19,683 configurations, L = 15
  the amplitudes, 2^(L + 1): 65,536 Toffolis
  the lookup, D: 19,683 Toffolis
  the erasure, (2L - 2) D: 551,124 Toffolis
  step 1's 15 rotations, (c - 3) L with c = 22 bits per angle: 285 Toffolis
the library counts 636,628 Toffolis; the bound (2L + 3) D = 649,539
recognizing whole configurations instead: (108 - 1) D = 2,106,081 Toffolis


## Loading a known state (iii): helium's trick, pin one nucleon

Where does the table come from? From a classical computation of the same Hamiltonian.
Helium-4 has exactly one nucleon of each species, so a configuration is four sites, one
for each nucleon, and on the $3^3$ lattice there are $27^4 = 531{,}441$ of them. The
Hamiltonian in that basis is a sparse matrix: a configuration connects to itself,
through the contact terms, and to the $24$ configurations one hop away, four nucleons
times six neighbours. Its lowest eigenvector is the ground state, and a laptop finds it
with a sparse eigensolver. The register sees the same state as a table of bit strings:
of the $2^{108}$ strings that $108$ mode qubits can hold, only those $531{,}441$ have
one nucleon of each species, and every one of them carries an amplitude. That is what
makes the load sparse: the register's size against the few strings that are physical;
nothing is truncated.

Pinning uses a symmetry to shrink the table. The Hamiltonian is the same at every site
and the ground state is unique, so translating the ground state by any lattice vector
gives the ground state back: it has zero total momentum. Then the $p\!\uparrow$ nucleon
is equally likely at each of the $27$ sites, and the ground state splits into $27$
pieces by where that nucleon sits, each of weight $1/27$, orthogonal to one another
because the nucleon is somewhere else in each, and each a translate of the piece with
the nucleon at the origin. That piece, normalized, is the pinned state: the other three
nucleons anywhere, $27^3 = 19{,}683$ configurations, `pinned_table_amplitudes`. Knowing
it is knowing the state, since the state is the sum of its $27$ translates over
$\sqrt{27}$, and loading it costs the six steps on a table $27$ times shorter.

The spread turns the pinned state into that sum. It needs a *shift register* of three
two-qubit registers, one per axis, each put into an equal superposition of the three
shifts $0$, $1$, $2$ along its axis. Three values are not a power of two, so Hadamards
do not do it: each axis takes one round of amplitude amplification, four Toffolis and
two rotations (Babbush et al., arXiv:1805.03662, Fig. 12). Controlled on each bit of
each axis register, the lattice is then shifted along that axis by one site or by two. A
shift of the whole lattice is a permutation of its mode qubits, done with controlled
swaps at one Toffoli each, and a cyclic shift of three blocks is two swaps of a third of
the lattice, $36$ qubits: $72$ controlled swaps per shift, six shifts, $432$.

Nucleons are fermions, and the register orders their modes along each line of the
lattice, so moving the last block of a line in front of the others changes the sign of
the state by $(-1)^{N_A N_C}$, with $N_A$ and $N_C$ the nucleons in the two groups of
blocks that pass each other. Two CNOT chains compute the two parities into two work
qubits, one controlled-controlled-Z applies the sign, and the chains run again: one
Toffoli per line per shift bit, $9 + 3 + 1 = 13$ lines along the three axes, $26$. After
the shifts the register holds the $27$ translates tagged by the shift, the situation of
the load after its step 2, and the tag must go. Here the reverse lookup is free, because
the tag is written in the configuration itself: the $p\!\uparrow$ species has $27$ mode
qubits and exactly one of them is $1$, at the site the shift moved the nucleon to, so
CNOTs from each of those qubits into the shift register, writing that site's
coordinates, XOR the shift onto itself and leave zero on every branch. No Toffoli and no
measurement; the step works only because the species holds one nucleon, since two would
XOR two sites together. Other nuclei need another pinning, which is the next section's
problem.

In all the spread costs $12 + 432 + 26 = 470$ Toffolis and $6$ rotations, $204$ $T$ at
the $34$ $T$ per rotation of the synthesis section, on $8$ ancillas the load has already
returned to zero: six for the shift register and two for the parities. Beside the load's
$636{,}628$ Toffolis it is nothing, and it buys a lot: loading the whole table of
$531{,}441$ configurations by the six steps would take an enumeration register of
$L = 20$ qubits and about $22.8$ million Toffolis, $36$ times the pinned load and spread
together. `helium_weak.load`, a `GroundStateLoad`, holds both counts; the next cell
shows the arithmetic.

In [51]:
helium_lattice = helium_weak.lattice
pinned = pinned_table_amplitudes(helium.nucleus, helium_lattice)
full_table = ground_state_amplitudes(helium.nucleus, helium_lattice)
assert full_table == helium_lattice.site_count * pinned
print(f"helium on {helium.box_side}^3: {helium_lattice.system_qubits} mode qubits")
print(
    f"  bit strings with one nucleon of each species: {full_table:,} "
    f"of 2^{helium_lattice.system_qubits}"
)
print(
    f"  {helium_lattice.site_count} translations, so the pinned table has "
    f"{pinned:,} amplitudes"
)
axis_superposition = equal_superposition_cost(helium.box_side)
shift_bits = axis_superposition.register_qubits
lattice_third = helium_lattice.system_qubits // helium.box_side
swaps_per_shift = (helium.box_side - 1) * lattice_third
lines_per_shift_bit = helium.box_side**2 + helium.box_side + 1
superposition_toffolis = 3 * axis_superposition.toffoli_count
swap_toffolis = 3 * shift_bits * swaps_per_shift
sign_toffolis = shift_bits * lines_per_shift_bit
spread_pieces = {
    "three equal superpositions of 3 values": superposition_toffolis,
    f"controlled swaps, {3 * shift_bits} shifts of {swaps_per_shift}": swap_toffolis,
    f"fermionic signs, {lines_per_shift_bit} lines, {shift_bits} bits": sign_toffolis,
}
assert sum(spread_pieces.values()) == helium_load.spread_toffolis
assert 3 * axis_superposition.rotation_count == helium_load.spread_rotations
spread_rotation_t = helium_load.spread_rotations * helium_weak.synthesis.t_per_rotation
print("the spread:")
for piece, piece_toffolis in spread_pieces.items():
    print(f"  {piece}: {piece_toffolis} Toffolis")
print(
    f"  {helium_load.spread_rotations} rotations at "
    f"{helium_weak.synthesis.t_per_rotation} T each: {spread_rotation_t} T"
)
print(
    f"{helium_load.load_toffolis:,} Toffolis to load, and "
    f"{helium_load.spread_toffolis} Toffolis and "
    f"{helium_load.spread_rotations} rotations to spread"
)
print(f"  = {helium_weak.load_t:,} T per load")
direct_load = ground_state_load(full_table, helium_lattice, helium_load.gradient_qubits)
pinned_and_spread = helium_load.load_toffolis + helium_load.spread_toffolis
saving = direct_load.load_toffolis / pinned_and_spread
print(
    f"loading the whole table instead, L = {direct_load.enumeration_qubits}: "
    f"{direct_load.load_toffolis:,} Toffolis, "
    f"{saving:.0f} times the pinned load and spread"
)

helium on 3^3: 108 mode qubits
  bit strings with one nucleon of each species: 531,441 of 2^108
  27 translations, so the pinned table has 19,683 amplitudes
the spread:
  three equal superpositions of 3 values: 12 Toffolis
  controlled swaps, 6 shifts of 72: 432 Toffolis
  fermionic signs, 13 lines, 2 bits: 26 Toffolis
  6 rotations at 34 T each: 204 T
636,628 Toffolis to load, and 470 Toffolis and 6 rotations to spread
  = 2,548,596 T per load
loading the whole table instead, L = 20: 22,823,731 Toffolis, 36 times the pinned load and spread


## The ground state is the biggest cost: helium's load, priced

Helium on the $3^3$ lattice is the case where the ground state can be priced. The
circuit loads it once for each preparation, $1 + 2m$ times (the amplification below
says why), so the loads, `all_loads_t`, outweigh everything else, and
`everything_priced_t` adds them to the rest. The lesson: the walk is the cheap part, and
the state is the cost.

In [52]:
for row in (helium_strong, helium_weak):
    share = 100 * row.all_loads_t / row.everything_priced_t
    print(
        f"{row.schedule.preparations} loads: {row.all_loads_t:.2e} T, "
        f"{share:.0f}% of the circuit"
    )
print(f"the energy measurement: {helium_weak.energy_measurement_t:.2e} T")
print(
    f"one circuit: {helium_strong.everything_priced_t:.3g} to "
    f"{helium_weak.everything_priced_t:.3g} T, "
    f"{100 * helium_strong.everything_priced_t / budget:.1f}% to "
    f"{100 * helium_weak.everything_priced_t / budget:.1f}% of the budget"
)

29 loads: 7.39e+07 T, 93% of the circuit
43 loads: 1.10e+08 T, 95% of the circuit
the energy measurement: 5.32e+06 T
one circuit: 7.93e+07 to 1.15e+08 T, 7.9% to 11.5% of the budget


## Carbon has no classical table

Can we load carbon's ground state from a classical table of its amplitudes? Helium's
trick fails. Carbon holds three nucleons of every species, so after the spread the three
$p\!\uparrow$ nucleons cannot tell which one was pinned: the shift register cannot be
erased, and it stays entangled with the system. Pinning would not be enough anyway. The
table would list every configuration, `ground_state_amplitudes`, or about $M$ times
fewer once the lattice's translations are used, `amplitudes_after_translations`: far
beyond any classical table, and beyond any load circuit, whose cost grows with the
table. With nothing pinned, the module has no table to load: `weak.load` is `None`,
carbon's preparation is unpriced, and the totals below are a floor on the circuit. The
ground state must be prepared on the machine, by an algorithm rather than from a table.

In [53]:
after = amplitudes_after_translations(nucleus, lattice)
print(f"carbon's load: {weak.load}")
print(f"amplitudes: {amplitudes:.2e}; the translations would leave about {after:.2e}")

carbon's load: None
amplitudes: 3.01e+18; the translations would leave about 4.71e+16


## Applying the current: a block encoding that rarely succeeds

This is step 2 of the recipe from hour 1. The ground state is in the register, loaded
from a table as helium's was or prepared by an algorithm as carbon's must be; now the
current acts on it, and step 3, the phase estimation of the walk, reads the energy of
one final state in what comes out.

The current is not unitary. Here
$J = \rho_{\vec q} = \sum_x e^{i\vec q\cdot\vec x}\, n_x$, the proton density at
momentum $\vec q$, summed over the two proton modes of every site. It is one
representative current; the back-up slide "The five response functions" lists what the
cross section needs. Applying it to the ground state is what the papers call the
*insertion*: the current operator inserted between the preparation and the energy
measurement. Its output, $J|\Psi_0\rangle$ normalized, is the state phase estimation
then reads. Written over the eigenstates $|f\rangle$ of $H$,
$J|\Psi_0\rangle = \sum_f |f\rangle\langle f|J|\Psi_0\rangle$, so the energy $E_f$ comes
out with probability $|\langle f|J|\Psi_0\rangle|^2/\eta^2$, and the histogram of the
energy transfers $E_f - E_0$ is the response function, up to its total weight $\eta^2$:
the quantity the whole calculation is for.

The current is block-encoded, as $H$ was, and a block encoding starts from the
operator's list of Pauli terms. Write $M$ for the number of lattice sites, as before:
$M = 4^3 = 64$ here. Each site has two proton modes, and a mode's occupation is
$n = (1 - Z)/2$, with $Z$ the Pauli operator on the mode's qubit ("From fermions to
qubits"), so

$$\rho_{\vec q} = \sum_x e^{i\vec q\cdot\vec x}\left(n_{x,p\uparrow} + n_{x,p\downarrow}\right) = \sum_x e^{i\vec q\cdot\vec x} - \frac{1}{2}\sum_x e^{i\vec q\cdot\vec x}\left(Z_{x,p\uparrow} + Z_{x,p\downarrow}\right).$$

The first sum is zero for every $\vec q \neq 0$: the phases cancel around the lattice.
What is left is $2M = 128$ terms, a Pauli $Z$ on each proton mode, each with a
coefficient of size $\tfrac{1}{2}$. The current's one-norm $\Lambda_J$ is the sum of
those sizes, as $\lambda$ is for $H$:

$$\Lambda_J = 2M \times \tfrac{1}{2} = M = 64.$$

So the one-norm equals the number of sites, which is why the slides write
$\Lambda_J = M$.

`weak.current` holds this block encoding's counts, the `Insertion` that the library's
`insertion` returns. Its index register holds $x$, $y$, $z$ and a spin bit, $7$ qubits.
Every term has the same size, so PREPARE is an equal superposition over the index, and
SELECT is a tree with one leaf per term: $2M - 1 = 127$ ANDs ("SELECT"). On the $4^3$
lattice every phase $e^{i\vec q\cdot\vec x}$ is a power of $i$, a Clifford gate on the
index's bits, so it takes no rotation. Every ancilla reads $0$, a *success*, with
probability (Roggero, Gu, Baroni and Papenbrock 2020, arXiv:2009.13485, Eq. 31)

$$P_s = \frac{\eta^2}{\Lambda_J^2}, \qquad \eta^2 = \langle\Psi_0|J^\dagger J|\Psi_0\rangle,$$

the rule of "Block encoding (iv)" with $\Lambda_J$ in the place of $\lambda$ and
$\|J|\Psi_0\rangle\|^2 = \eta^2$. The strength $\eta^2$ is the response's total weight
at $\vec q$, every energy transfer added up. The nucleus fixes it, but for carbon we
have no value: we take it between half of carbon's six protons and all of them, $3$ to
$6$, the strength band, an estimate. The back-up slide "The strength $\eta^2$: what it
is, and why we take $[Z/2, Z]$" gives the reasons. So $P_s$ lies between
$3/64^2 \approx 7.3 \times 10^{-4}$ and $6/64^2 \approx 1.5 \times 10^{-3}$: about one
success in a thousand attempts.

The insertion sits here, between the ground state and the sum of the costs, because its
failures are what multiply the ground state's price: a run that fails has spent its
preparation, and the next section rescues the failures by amplitude amplification, every
round of which undoes and then redoes the preparation and the insertion: two more of
each per round.

In [54]:
current = weak.current
print(
    f"the insertion: {current.terms:,} terms, {current.index_qubits} index "
    f"qubits, {current.toffolis:,} Toffolis, {current.rotations:,} rotations, "
    f"{current.t_gates} T gates"
)
print(f"one insertion: {weak.insertion_t:,} T")
assert current.terms / 2 == lattice.site_count
print(
    f"the current's one-norm: {current.terms} terms of size 1/2, "
    f"{current.terms / 2:.0f}, the number of sites"
)
assert math.isclose(weak.schedule.success_probability, weakest / lattice.site_count**2)
assert math.isclose(
    strong.schedule.success_probability, strongest / lattice.site_count**2
)
print(
    f"P_s: {weak.schedule.success_probability:.2e} to "
    f"{strong.schedule.success_probability:.2e}"
)

the insertion: 128 terms, 7 index qubits, 127 Toffolis, 0 rotations, 0 T gates
one insertion: 508 T
the current's one-norm: 128 terms of size 1/2, 64, the number of sites
P_s: 7.32e-04 to 1.46e-03


The notebook's worked example so far has been one site. This section and the next leave
it, for a reason: one site has no momentum to transfer. Its only momentum is
$\vec q = 0$, where the density is just the number of protons on the site, and the
insertion would only multiply the state by that number. Carbon's own insertion is far
too large to simulate. So the cells below work on the smallest piece of the lecture's
lattice that shows the current at work: one row of it, four sites on a ring, holding one
proton. Only the proton modes are kept, since the current does not touch a neutron.
Everything said above carries over at this size: $M = 4$ sites with two proton modes
each, so $2M = 8$ terms of size $\tfrac{1}{2}$ and $\Lambda_J = M = 4$. The momentum is
a single number, and the cells take the smallest the ring has above zero, $q = \pi/2$
per site, so the phase on site $x$ is $e^{iqx} = i^x$, a power of $i$ as on the $4^3$
lattice.

The ring's ground state is the proton at rest, taken here with spin up: amplitude
$\tfrac{1}{2}$ on each of the four sites. As a table that is $D = 4$ configurations, and
the next cell loads it by the steps of "Loading a known state". Two `row` qubits take
equal amplitudes on the four row numbers (step 1), and each row number writes the proton
onto its site (step 2). The erasure is cheap for this table. Row $x$ puts the proton on
site $x$, so the proton's position spells the row number in binary, and a CNOT from a
site's qubit into each `row` qubit that is 1 in that site's number erases it: CNOTs
alone, with no identifier. The cell checks the state it loads. It also checks that this
state has the lowest kinetic energy a proton can have on the ring, taking the kinetic
energy as $2t$ less $t$ times the hops to the two neighbors, the first two terms of $H$
in one dimension.

In [55]:
ring_sites = 4
ring_up = cirq.NamedQubit.range(ring_sites, prefix="p_up")
ring_down = cirq.NamedQubit.range(ring_sites, prefix="p_down")
ring_modes = [*ring_up, *ring_down]
ring_row = [cirq.NamedQubit("row1"), cirq.NamedQubit("row0")]
ring_load = cirq.Circuit(cirq.H.on_each(*ring_row))
for ring_site in range(ring_sites):
    ring_digits = [int(digit) for digit in format(ring_site, "02b")]
    ring_load.append(
        cirq.X(ring_up[ring_site]).controlled_by(*ring_row, control_values=ring_digits)
    )
for ring_site in range(ring_sites):
    ring_digits = [int(digit) for digit in format(ring_site, "02b")]
    for ring_digit, ring_row_qubit in zip(ring_digits, ring_row, strict=True):
        if ring_digit:
            ring_load.append(cirq.Moment(cirq.CNOT(ring_up[ring_site], ring_row_qubit)))
print(ring_load.to_text_diagram(qubit_order=[*ring_row, *ring_up]))
ring_proton_on = [
    cirq.final_state_vector(
        cirq.Circuit(cirq.X(mode)), qubit_order=ring_modes, dtype=np.complex128
    )
    for mode in ring_up
]
ring_at_rest = sum(ring_proton_on) / 2
ring_loaded = cirq.final_state_vector(
    ring_load, qubit_order=[*ring_modes, *ring_row], dtype=np.complex128
).reshape(256, 4)
assert np.allclose(ring_loaded[:, 0], ring_at_rest)
assert np.allclose(ring_loaded[:, 1:], 0)
print("loaded: amplitude 1/2 with the proton on each site, the row register back at 0")
ring_shift = np.roll(np.eye(ring_sites), 1, axis=0)
ring_kinetic = LECTURE_COUPLINGS.hopping_scale_mev * (
    2 * np.eye(ring_sites) - ring_shift - ring_shift.T
)
ring_energies, ring_kinetic_states = np.linalg.eigh(ring_kinetic)
assert np.allclose(np.abs(ring_kinetic_states[:, 0]), 0.5)
print(
    "one proton's kinetic energies on the ring, in MeV: "
    + ", ".join(f"{energy:.2f}" for energy in np.round(ring_energies, 2) + 0.0)
)
print("the lowest is the proton at rest")

row1: ────H───(0)───(0)───@─────@───────X───X───────
              │     │     │     │       │   │
row0: ────H───(0)───@─────(0)───@───X───┼───┼───X───
              │     │     │     │   │   │   │   │
p_up0: ───────X─────┼─────┼─────┼───┼───┼───┼───┼───
                    │     │     │   │   │   │   │
p_up1: ─────────────X─────┼─────┼───@───┼───┼───┼───
                          │     │       │   │   │
p_up2: ───────────────────X─────┼───────@───┼───┼───
                                │           │   │
p_up3: ─────────────────────────X───────────@───@───
loaded: amplitude 1/2 with the proton on each site, the row register back at 0
one proton's kinetic energies on the ring, in MeV: 0.00, 21.16, 21.16, 42.32
the lowest is the proton at rest


Next the insertion on the ring, with the pieces the prose above lists. Its index
register has three qubits: `x1` and `x0` hold the site number $x$ in binary, and `spin`
picks one of the site's two proton modes. PREPARE is a Hadamard on each index qubit,
since every term has the same size. The phase $i^x$ is two Clifford gates on the bits of
$x$: a Pauli $Z$ on `x1`, which gives $-1$ when that bit is 1, and an $S$ on `x0`, which
gives $i$. SELECT applies a Pauli $Z$ to the proton mode the index names. Here that is
eight gates, each controlled on all three index qubits; on the machine it is a tree of
$2M - 1 = 7$ ANDs. A global sign supplies the minus of $-\tfrac{1}{2}$, and the
Hadamards undo PREPARE.

The next cell first writes the current out with Cirq, site by site, so that the identity
terms and their cancellation can be seen. It then builds the circuit and checks the rule
of "Block encoding (iv)": with the index at $0$ on the way in and on the way out, the
circuit is $\rho_q/\Lambda_J$. The cell after it runs the load and then the insertion,
13 qubits in all, and reads the state. The index reads $000$ with probability
$\eta^2/\Lambda_J^2 = 1/16$; for a single proton the strength is $\eta^2 = 1$, its
proton count, the top of the strength band. In the success part the proton has taken up
the momentum: its amplitude on site $x$ is $i^x/2$, a plane wave, with kinetic energy
$2t$. On the ring that one energy is the whole response at this $q$, the energy step 3
of the recipe would read.

In [56]:
ring_density = cirq.PauliSum()
print("the current on the ring, site by site:")
for ring_site in range(ring_sites):
    ring_protons = (
        cirq.PauliString()
        - 0.5 * cirq.Z(ring_up[ring_site])
        - 0.5 * cirq.Z(ring_down[ring_site])
    )
    print(f"  site {ring_site}: {1j**ring_site * ring_protons}")
    ring_density += 1j**ring_site * ring_protons
ring_one_norm = sum(abs(term.coefficient) for term in ring_density)
assert len(ring_density) == 2 * ring_sites
assert all(math.isclose(abs(term.coefficient), 0.5) for term in ring_density)
assert math.isclose(ring_one_norm, ring_sites)
print(f"summed, the four I terms cancel and {len(ring_density)} terms of size 1/2 stay")
print(f"their one-norm: {ring_one_norm:.0f}, the number of sites")
ring_index = [cirq.NamedQubit("x1"), cirq.NamedQubit("x0"), cirq.NamedQubit("spin")]
ring_insertion = cirq.Circuit(cirq.H.on_each(*ring_index))
ring_insertion.append([cirq.Z(ring_index[0]), cirq.S(ring_index[1])])
for ring_site in range(ring_sites):
    ring_digits = [int(digit) for digit in format(ring_site, "02b")]
    for ring_spin, ring_species in enumerate((ring_up, ring_down)):
        ring_insertion.append(
            cirq.Z(ring_species[ring_site]).controlled_by(
                *ring_index, control_values=[*ring_digits, ring_spin]
            )
        )
ring_insertion.append(cirq.global_phase_operation(-1))
ring_insertion.append(cirq.H.on_each(*ring_index))
print(ring_insertion.to_text_diagram(qubit_order=[*ring_index, *ring_modes]))
ring_density_matrix = ring_density.matrix(ring_modes)
ring_block = ring_insertion.unitary(qubit_order=[*ring_index, *ring_modes])[:256, :256]
assert np.allclose(ring_block, ring_density_matrix / ring_one_norm)
print("with the index at 0 on the way in and out, the circuit is the current / 4")

the current on the ring, site by site:
  site 0: 1.0*I-0.5*Z(p_up0)-0.5*Z(p_down0)
  site 1: 1.0j*I-0.5j*Z(p_up1)-0.5j*Z(p_down1)
  site 2: -1.0*I+0.5*Z(p_up2)+0.5*Z(p_down2)
  site 3: -1.0j*I+0.5j*Z(p_up3)+0.5j*Z(p_down3)
summed, the four I terms cancel and 8 terms of size 1/2 stay
their one-norm: 4, the number of sites
x1: ────────────H───Z───(0)───(0)───(0)───(0)───@─────@─────@─────@───H───
                        │     │     │     │     │     │     │     │
x0: ────────────H───S───(0)───(0)───@─────@─────(0)───(0)───@─────@───H───
                        │     │     │     │     │     │     │     │
spin: ──────────H───────(0)───@─────(0)───@─────(0)───@─────(0)───@───H───
                        │     │     │     │     │     │     │     │
p_up0: ─────────────────Z─────┼─────┼─────┼─────┼─────┼─────┼─────┼───────
                              │     │     │     │     │     │     │
p_up1: ───────────────────────┼─────Z─────┼─────┼─────┼─────┼─────┼───────
                              

In [57]:
ring_qubits = [*ring_index, *ring_modes, *ring_row]
ring_start = ring_load + ring_insertion
ring_inserted = cirq.final_state_vector(
    ring_start, qubit_order=ring_qubits, dtype=np.complex128
).reshape(8, 256, 4)
ring_success = np.sum(np.abs(ring_inserted[0]) ** 2)
ring_strength = np.linalg.norm(ring_density_matrix @ ring_at_rest) ** 2
assert math.isclose(ring_success, ring_strength / ring_one_norm**2)
print(f"the strength of one proton: {ring_strength:.2f}")
print(
    f"the index reads 000 with probability {ring_success:.4f}, "
    "the strength / one-norm^2"
)
ring_wave = np.array(
    [np.vdot(proton_there, ring_inserted[0, :, 0]) for proton_there in ring_proton_on]
) / math.sqrt(ring_success)
assert np.allclose(ring_wave, [1j**x / 2 for x in range(ring_sites)])
print("the success part: amplitudes 1/2, i/2, -1/2, -i/2 on sites 0 to 3")
ring_kick_energy = 2 * LECTURE_COUPLINGS.hopping_scale_mev
assert np.allclose(ring_kinetic @ ring_wave, ring_kick_energy * ring_wave)
print(
    "that is the proton with one unit of momentum, kinetic energy 2t = "
    f"{ring_kick_energy:.2f} MeV"
)

the strength of one proton: 1.00
the index reads 000 with probability 0.0625, the strength / one-norm^2
the success part: amplitudes 1/2, i/2, -1/2, -i/2 on sites 0 to 3
that is the proton with one unit of momentum, kinetic energy 2t = 21.16 MeV


## Amplitude amplification: making a rare success near certain

After the preparation and the insertion the index register reads $0$ or it does not: the
two outcomes of "throw the leftover away" in "Block encoding's two uses". Group the
state by them,
$\sin\theta\,|\mathrm{success}\rangle + \cos\theta\,|\mathrm{failure}\rangle$, where the
success part is $J|\Psi_0\rangle$ normalized, with the index at $0$, and the failure
part is everything else. The weight of success is $P_s$, so $\sin\theta = \sqrt{P_s}$;
for carbon $\theta$ is between $1.5$ and $2.2$ degrees, and the arrow starts nearly flat
along the failure axis. Trying and discarding, one preparation and insertion per attempt
with the index measured, would take $1/P_s$ preparations per success, about $680$ to
$1{,}370$, each of them a ground state.

Amplitude amplification (Brassard, Hoyer, Mosca and Tapp 2002, arXiv:quant-ph/0005055;
Grover's search is the special case) turns the arrow instead of rolling the dice. A
round is two mirrors, the geometry of "Why the walk encodes the energy". Mirror 1 is the
failure axis: flip the sign of the success part, which is a sign conditioned on the
index register reading $0$, an AND chain over the $7$ index qubits. Mirror 2 is the
start line, the line through the arrow the preparation and insertion made: the circuit
makes that mirror by undoing the preparation and the insertion, flipping the sign of
every state but the all-zero one, an AND chain over the $263$ system and index qubits,
and redoing them. Two mirrors $\theta$ apart turn every arrow by $2\theta$, just as the
walk's two mirrors $\theta_E/2$ apart turned the arrow in the pair's plane by
$\theta_E$; here the turn is toward success. After $m$ rounds the arrow is at
$(2m + 1)\theta$, so $m \approx \pi/(4\sqrt{P_s})$ rounds bring it near a quarter turn,
and the undo and redo of every round is where the $1 + 2m$ preparations come from.

The strength is not known in advance, so `Nucleus.strength_band` takes the range
$[Z/2, Z]$ as an estimate. `amplification` takes the $m$ that lands nearest a quarter
turn, the textbook rule, and `weak.schedule` is the `Amplification` it returns. Its
landing probability, $\sin^2((2m + 1)\theta)$, is a little below 1. A circuit that
misses is run again; the miss is found at the index, before the energy measurement. Each
round's two reflections are AND chains, `weak.reflections`. The next cell prints
carbon's numbers. The cells after it run the rounds themselves, on the ring of "Applying
the current".

In [58]:
weakest, strongest = nucleus.strength_band
weak_schedule = weak.schedule
strong_schedule = strong.schedule
print(f"strength band: {weakest} to {strongest}")
print(
    f"P_s: {weak_schedule.success_probability:.2e} to "
    f"{strong_schedule.success_probability:.2e}"
)
print(
    f"pi / (4 sqrt(P_s)): "
    f"{math.pi / (4 * math.sqrt(strong_schedule.success_probability)):.1f} to "
    f"{math.pi / (4 * math.sqrt(weak_schedule.success_probability)):.1f}"
)
print(f"rounds: {strong_schedule.rounds} to {weak_schedule.rounds}")
print(f"preparations: {strong_schedule.preparations} to {weak_schedule.preparations}")
print(
    f"landing probability: {weak_schedule.landing_probability:.6f} to "
    f"{strong_schedule.landing_probability:.6f}"
)
print(
    "preparations per success, as a circuit that misses runs again: "
    f"{strong_schedule.preparations_per_success:.2f} to "
    f"{weak_schedule.preparations_per_success:.2f}"
)
print(
    f"one round's reflections: {weak.reflections.index_chain_ands:,} + "
    f"{weak.reflections.start_chain_ands:,} ANDs = {weak.round_reflections_t:,} T"
)

strength band: 3.0 to 6.0
P_s: 7.32e-04 to 1.46e-03
pi / (4 sqrt(P_s)): 20.5 to 29.0
rounds: 20 to 29
preparations: 41 to 59
landing probability: 0.999317 to 0.999999
preparations per success, as a circuit that misses runs again: 41.00 to 59.04
one round's reflections: 5 + 261 ANDs = 1,064 T


Carbon's rounds cannot be simulated, so the last two cells of this section amplify the
ring's insertion, with the registers and circuits of "Applying the current". The start
state is the load followed by the insertion, on 13 qubits, and its success part, where
the index reads $000$, has weight $1/16$. Nothing below moves the state with the
rotation formula. The cells build the two mirrors as circuits, apply them to the
simulated state, and only then compare what they read off the state with
$\sin^2((2m + 1)\theta)$.

Mirror 1 flips the sign of the success part. On the circuit that is a sign on one value
of the index register, $000$: an $X$ on each index qubit turns $000$ into $111$, a Pauli
$Z$ on one of them controlled on the other two flips the sign of $111$ alone, and the
$X$s turn it back.

Mirror 2 reflects about the start state: it keeps the start state and flips the sign of
everything at right angles to it. No short circuit singles out a state like that. The
one state a circuit can single out cheaply is all zeros, by mirror 1's trick over every
qubit. So mirror 2 takes three moves: undo the insertion and the load, which carries the
start state back to all zeros; flip the sign of everything but the all-zero state; redo
the load and the insertion. With $U$ for the load followed by the insertion, so that
$|\mathrm{start}\rangle = U|0\rangle$,

$$U\,\bigl(2|0\rangle\langle 0| - 1\bigr)\,U^\dagger = 2|\mathrm{start}\rangle\langle\mathrm{start}| - 1.$$

This is where the bill comes from: every round undoes the load once and redoes it once,
so $m$ rounds take $1 + 2m$ loads. The flip in the middle tests all 13 qubits, the `row`
register included, since any of them may be away from 0 once the load and the insertion
are undone on a state other than the start state. Cirq applies each many-controlled $Z$
as one gate; on the machine it is an AND chain, as REFLECT's was.

The first cell builds both mirrors and tries each on the start state: mirror 1 changes
the sign of the success part and leaves the rest, and mirror 2 changes nothing, since
the start state lies in it. The second cell applies rounds, mirror 1 then mirror 2, and
after each reads the simulated state: the chance that the index reads $000$; the state's
components along the success direction and the failure direction, which give the arrow's
angle; and whether those two components still hold all of the state, that is, whether
the rounds have stayed in the plane of the slide's figure. It counts the loads in each
circuit by the Hadamard that every load, done or undone, puts on `row1`. The number of
rounds comes from the library's rule, `amplification`, the rule that sets carbon's
rounds, and the cell checks the rule's landing probability against the simulated state.
The loop runs one round past the rule, to show the overshoot: beyond a quarter turn the
chance falls again, which is why the number of rounds has to be set from $P_s$ in
advance.

In [59]:
ring_flip_success = cirq.Circuit(
    cirq.X.on_each(*ring_index),
    cirq.Z(ring_index[2]).controlled_by(*ring_index[:2]),
    cirq.X.on_each(*ring_index),
)
ring_flip_all_but_zero = cirq.Circuit(
    cirq.X.on_each(*ring_qubits),
    cirq.Z(ring_qubits[-1]).controlled_by(*ring_qubits[:-1]),
    cirq.X.on_each(*ring_qubits),
    cirq.global_phase_operation(-1),
)
ring_start_mirror = cirq.inverse(ring_start) + ring_flip_all_but_zero + ring_start
print("mirror 1, on the index register:")
print(ring_flip_success.to_text_diagram(qubit_order=ring_index))
print("mirror 2's middle, the flip of all but the all-zero state:")
print(ring_flip_all_but_zero.to_text_diagram(qubit_order=ring_qubits))
ring_after_mirror_1 = cirq.final_state_vector(
    ring_start + ring_flip_success, qubit_order=ring_qubits, dtype=np.complex128
).reshape(8, 256, 4)
assert np.allclose(ring_after_mirror_1[0], -ring_inserted[0])
assert np.allclose(ring_after_mirror_1[1:], ring_inserted[1:])
print("mirror 1 on the start state: the success part changes sign, the rest stays")
ring_after_mirror_2 = cirq.final_state_vector(
    ring_start + ring_start_mirror, qubit_order=ring_qubits, dtype=np.complex128
).reshape(8, 256, 4)
assert np.allclose(ring_after_mirror_2, ring_inserted)
print("mirror 2 on the start state: no change, since the start state lies in it")

mirror 1, on the index register:
x1: ─────X───@───X───
             │
x0: ─────X───@───X───
             │
spin: ───X───@───X───
mirror 2's middle, the flip of all but the all-zero state:
x1: ────────────X───@───X───
                    │
x0: ────────────X───@───X───
                    │
spin: ──────────X───@───X───
                    │
p_up0: ─────────X───@───X───
                    │
p_up1: ─────────X───@───X───
                    │
p_up2: ─────────X───@───X───
                    │
p_up3: ─────────X───@───X───
                    │
p_down0: ───────X───@───X───
                    │
p_down1: ───────X───@───X───
                    │
p_down2: ───────X───@───X───
                    │
p_down3: ───────X───@───X───
                    │
row1: ──────────X───@───X───
                    │
row0: ──────────X───Z───X───

global phase:   π
mirror 1 on the start state: the success part changes sign, the rest stays
mirror 2 on the start state: no change, since the start state lies in it


In [60]:
ring_theta = math.asin(math.sqrt(ring_success))
ring_success_part = np.zeros_like(ring_inserted)
ring_success_part[0] = ring_inserted[0]
ring_success_direction = ring_success_part.ravel() / math.sin(ring_theta)
ring_failure_part = ring_inserted - ring_success_part
ring_failure_direction = ring_failure_part.ravel() / math.cos(ring_theta)
ring_schedule = amplification(ring_success)
ring_round = ring_flip_success + ring_start_mirror
ring_circuit = ring_start
ring_chances = []
print("rounds  loads  angle in degrees  chance that the index reads 000")
for ring_rounds in range(ring_schedule.rounds + 2):
    ring_loads = sum(
        1
        for operation in ring_circuit.all_operations()
        if operation.qubits == (ring_row[0],) and operation.gate == cirq.H
    )
    assert ring_loads == 1 + 2 * ring_rounds
    ring_state = cirq.final_state_vector(
        ring_circuit, qubit_order=ring_qubits, dtype=np.complex128
    )
    ring_along_success = np.vdot(ring_success_direction, ring_state)
    ring_along_failure = np.vdot(ring_failure_direction, ring_state)
    ring_in_plane = abs(ring_along_success) ** 2 + abs(ring_along_failure) ** 2
    assert math.isclose(ring_in_plane, 1)
    ring_angle = math.atan2(ring_along_success.real, ring_along_failure.real)
    assert math.isclose(ring_angle, (2 * ring_rounds + 1) * ring_theta)
    ring_chances.append(np.sum(np.abs(ring_state.reshape(8, 256, 4)[0]) ** 2))
    assert math.isclose(ring_chances[ring_rounds], math.sin(ring_angle) ** 2)
    print(
        f"{ring_rounds:6d}  {ring_loads:5d}  {math.degrees(ring_angle):16.1f}  "
        f"{ring_chances[ring_rounds]:.4f}"
    )
    ring_circuit = ring_circuit + ring_round
ring_landing = ring_chances[ring_schedule.rounds]
assert math.isclose(ring_landing, ring_schedule.landing_probability)
print(
    f"the library's rule for P_s = {ring_success:.4f}: {ring_schedule.rounds} rounds, "
    f"landing probability {ring_schedule.landing_probability:.4f}"
)
print(
    f"loads per success: {ring_schedule.preparations_per_success:.2f} amplified, "
    f"{1 / ring_success:.0f} by trying and discarding"
)

rounds  loads  angle in degrees  chance that the index reads 000
     0      1              14.5  0.0625
     1      3              43.4  0.4727
     2      5              72.4  0.9084
     3      7             101.3  0.9613
     4      9             130.3  0.5817
the library's rule for P_s = 0.0625: 3 rounds, landing probability 0.9613
loads per success: 7.28 amplified, 16 by trying and discarding


The textbook rule turns the arrow as near to certainty as it can. Is that the cheapest
place to stop? The table above lets us check. A circuit that misses is run again, so
what a success costs on average is the loads in one circuit divided by the chance that
it lands. On the ring that is $1/0.0625 = 16$ loads with no rounds, $3/0.4727 = 6.3$
after one round, $5/0.9084 = 5.5$ after two, and $7/0.9613 = 7.3$ after the rule's
three. Two rounds come cheaper than three: the third round lifts the chance from $0.91$
to $0.96$ and adds two loads to every circuit, and the lift does not pay for them.

Carbon tells the same story. At the weak end of the strength band the rule's $29$ rounds
land with probability $0.9993$ and take $59$ preparations a success. Stopping at $21$
rounds lands only 84% of the time and takes $51$, in a circuit of $43$ preparations in
place of $59$, so the circuit is shallower as well. The reason is the shape of
$\sin^2((2m + 1)\theta)$: near the quarter turn it flattens, while every round still
costs two preparations. With $x = (2m + 1)\theta$ for the angle at the stop, the average
cost goes as $x/\sin^2 x$, which is lowest where $\tan x = 2x$, near $67$ degrees: three
quarters of the way to the quarter turn, whatever $\theta$ is. The next cell finds the
cheapest stop, for the ring from the chances the simulation measured, and for carbon
from the formula those chances confirmed.

Two lessons here reach beyond this circuit. A textbook solution is a good starting
point, and often not the best option once you dig around. And maximizing along one axis
can mislead: going to maximum certainty sounds optimal, and it is the more expensive
choice on average when a miss can be tried again.

The lecture's counts keep the textbook rule, which is the simpler story to tell: one
circuit that almost always lands. The cheapest stop would take 12 to 14% off carbon's
preparations per success. The back-up slide "Step 2's trade: many short circuits, or one
deep one" turns the same dial against the machine: how many rounds its logical error
rate can afford, and what the missing rounds cost in wall time. "Wall time" prints its
table.

In [61]:
print("the ring, from the chances the simulation measured:")
print("rounds  loads  chance  loads per success")
ring_costs = []
for ring_stop, ring_chance in enumerate(ring_chances):
    ring_costs.append((1 + 2 * ring_stop) / ring_chance)
    print(
        f"{ring_stop:6d}  {1 + 2 * ring_stop:5d}  {ring_chance:.4f}  "
        f"{ring_costs[ring_stop]:17.2f}"
    )
ring_cheapest = ring_costs.index(min(ring_costs))
assert ring_cheapest < ring_schedule.rounds
print(
    f"cheapest stop: {ring_cheapest} rounds, at "
    f"{math.degrees((1 + 2 * ring_cheapest) * ring_theta):.1f} degrees; "
    f"the rule takes {ring_schedule.rounds}"
)
print("carbon, from sin^2((2m + 1) theta), which those chances confirmed:")
print("                      rounds  preparations  chance  per success")
for stop_end, stop_rule in (
    ("weak end", weak_schedule),
    ("strong end", strong_schedule),
):
    stop_theta = math.asin(math.sqrt(stop_rule.success_probability))
    stop_costs = []
    for stop_rounds in range(stop_rule.rounds + 1):
        stop_angle = (1 + 2 * stop_rounds) * stop_theta
        stop_costs.append((1 + 2 * stop_rounds) / math.sin(stop_angle) ** 2)
    assert math.isclose(stop_costs[-1], stop_rule.preparations_per_success)
    stop_cheapest = stop_costs.index(min(stop_costs))
    assert stop_cheapest < stop_rule.rounds
    stop_angle = (1 + 2 * stop_cheapest) * stop_theta
    print(
        f"{stop_end:>10}  the rule  {stop_rule.rounds:6d}  "
        f"{stop_rule.preparations:12d}  {stop_rule.landing_probability:.4f}  "
        f"{stop_costs[-1]:11.2f}"
    )
    print(
        f"{'':>10}  cheapest  {stop_cheapest:6d}  {1 + 2 * stop_cheapest:12d}  "
        f"{math.sin(stop_angle) ** 2:.4f}  {stop_costs[stop_cheapest]:11.2f}"
    )
    print(
        f"{'':>10}  the cheapest stop is at {math.degrees(stop_angle):.1f} degrees "
        f"and saves {1 - stop_costs[stop_cheapest] / stop_costs[-1]:.0%}"
    )
stop_grid = np.linspace(0.5, 1.5, 100_001)
stop_lowest = stop_grid[np.argmin(stop_grid / np.sin(stop_grid) ** 2)]
assert math.isclose(math.tan(stop_lowest), 2 * stop_lowest, rel_tol=1e-3)
print(
    f"x / sin^2 x is lowest at {math.degrees(stop_lowest):.1f} degrees, "
    "where tan x = 2x"
)

the ring, from the chances the simulation measured:
rounds  loads  chance  loads per success
     0      1  0.0625              16.00
     1      3  0.4727               6.35
     2      5  0.9084               5.50
     3      7  0.9613               7.28
     4      9  0.5817              15.47
cheapest stop: 2 rounds, at 72.4 degrees; the rule takes 3
carbon, from sin^2((2m + 1) theta), which those chances confirmed:
                      rounds  preparations  chance  per success
  weak end  the rule      29            59  0.9993        59.04
            cheapest      21            43  0.8433        50.99
            the cheapest stop is at 66.7 degrees and saves 14%
strong end  the rule      20            41  1.0000        41.00
            cheapest      15            31  0.8596        36.06
            the cheapest stop is at 68.0 degrees and saves 12%
x / sin^2 x is lowest at 66.8 degrees, where tan x = 2x


## Everything we can price, added up

What does the circuit cost, as far as it can be priced? At the weak end, the costlier:
the energy measurement once, an insertion with each preparation, and two reflections in
each round. `everything_priced_t` adds them up, with the loads and their gradient state
where the preparation is priced. Carbon's is not, so `all_loads_t` is `None` and this
total is a floor on the circuit. The priced parts alone also set the error each hard
operation may make (the error-budget sections below).

In [62]:
priced = weak.everything_priced_t
preparations = weak_schedule.preparations
rounds = weak_schedule.rounds
print(
    f"the energy measurement: {measurement:,} T, "
    f"{100 * measurement / budget:.3g}% of the budget"
)
print(
    f"insertions, {weak.insertion_t:,} T x {preparations}: "
    f"{weak.insertions_t:,} T, {100 * weak.insertions_t / budget:.3g}% of the budget"
)
print(
    f"reflections, {weak.round_reflections_t:,} T x {rounds}: "
    f"{weak.reflections_t:,} T, {100 * weak.reflections_t / budget:.3g}% of the budget"
)
print(f"all the loads: {weak.all_loads_t}")
print(
    f"everything priced: {priced:,} T ({priced:.2e}), "
    f"{100 * priced / budget:.3g}% of the budget"
)

the energy measurement: 21,746,056 T, 2.17% of the budget
insertions, 508 T x 59: 29,972 T, 0.003% of the budget
reflections, 1,064 T x 29: 30,856 T, 0.00309% of the budget
all the loads: None
everything priced: 21,806,884 T (2.18e+07), 2.18% of the budget


## The allowance: what each preparation may cost

How much may carbon's unpriced preparation cost, for the circuit to fit the 2033
budget? `left_over_t` is the budget less everything but the preparations, and
`allowance_t` shares it over the $1 + 2m$ preparations, run forward or backward inside
the amplification, both directions at the same cost:

$$\text{allowance} = \frac{10^9 - \text{everything priced}}{1 + 2m}.$$

The circuit fits the 2033 budget if a preparation method meets it. The next cell does
the division at both ends of the strength band. The weak end takes more rounds, so more
preparations share the leftover and each may cost less. A preparation run backward costs
what it costs forward, since undoing a circuit runs the same gates in reverse, each one
inverted.

For scale, set the allowance against a preparation we did price. Helium's whole load,
the table of $19{,}683$ amplitudes and the spreading, priced in "The ground state is the
biggest cost: helium's load, priced", is `helium_weak.load_t`. Carbon's allowance is $7$
to $9$ times that, while carbon's table, counted in "Carbon has no classical table",
would be about $10^{12}$ times as long as helium's. A table is no way to spend the
allowance.

In [63]:
print(f"the budget: {budget:,} T")
print("            everything priced    left over  preparations  per preparation")
for allowance_end, allowance_row in (("weak end", weak), ("strong end", strong)):
    assert math.isclose(
        allowance_row.allowance_t,
        (budget - allowance_row.everything_priced_t)
        / allowance_row.schedule.preparations,
    )
    print(
        f"{allowance_end:>10}  {allowance_row.everything_priced_t:17,}  "
        f"{allowance_row.left_over_t:11,}  {allowance_row.schedule.preparations:12d}  "
        f"{allowance_row.allowance_t:15,.0f}"
    )
print(
    f"helium's whole load: {helium_weak.load_t:,} T; carbon's allowance is "
    f"{weak.allowance_t / helium_weak.load_t:.1f} to "
    f"{strong.allowance_t / helium_weak.load_t:.1f} times that"
)
print(
    f"carbon's table would be {after / helium_weak.load.amplitudes:.1e} times as "
    "long as helium's"
)

the budget: 1,000,000,000 T
            everything priced    left over  preparations  per preparation
  weak end         21,806,884  978,193,116            59       16,579,544
strong end         21,788,164  978,211,836            41       23,858,825
helium's whole load: 2,548,596 T; carbon's allowance is 6.5 to 9.4 times that
carbon's table would be 2.4e+12 times as long as helium's


## Why carbon's preparation is still unpriced: our view

The algorithms exist. An adiabatic path, a variational circuit and projection from a
trial state are well understood, and their costs are theorems or close to it. Their
inputs do not: each price needs numbers about carbon in this model, the gap along an
adiabatic path, or the gap $\Delta$ above the ground state and the overlap $\gamma$ of a
trial state we can afford to load. Nobody has computed them for this Hamiltonian on this
lattice.

A rule of thumb, ours. Projection from a trial state takes about
$\lambda/(\gamma\Delta)$ uses of $H$'s block encoding, up to a logarithm (Lin and Tong,
Quantum 4, 372 (2020), Thm. 6). That circuit is one we have priced. One use of $H$'s
block encoding with its reflection, that is PREPARE, SELECT, PREPARE's inverse and
REFLECT, is what step 3 runs as a walk step, at `weak.walk_step_t`. Step 3's walk is
paid for in full in "The walk and the readout, priced", and the allowance buys no more
of it. A preparation by projection would run the same circuit again, inside step 1, and
pay for it out of the allowance. So the allowance divided by one use's price,
`allowance_in_walk_steps`, is the number of uses each preparation can afford, and the
circuit fits only if $\gamma\Delta$ is at least $\lambda$ over that number,
`rule_of_thumb_mev`.

How the budget counts a Toffoli matters here. The module charges it 4 $T$, and
`one_per_toffoli` gives the same counts if the budget counts a Toffoli as one hard
operation, which pays for about four times the uses. The next two cells give the uses
and the rule of thumb both ways of counting, and the gap the rule asks for at an overlap
of one half. Pricing a preparation under this allowance is the open research problem of
this calculation.

In [64]:
weak_counted = weak.one_per_toffoli
strong_counted = strong.one_per_toffoli
print(f"one use of H's block encoding, with its reflection: {weak.walk_step_t:,} T")
print(
    "uses the allowance pays for in each preparation: "
    f"{weak.allowance_in_walk_steps:,.0f} to {strong.allowance_in_walk_steps:,.0f}"
)
print(f"for comparison, step 3's walk makes {register.walk_steps:,}")
print(
    "if the budget counts a Toffoli as one hard operation instead of 4 T: "
    f"{weak_counted.allowance_in_walk_steps:,.0f} to "
    f"{strong_counted.allowance_in_walk_steps:,.0f} uses"
)

one use of H's block encoding, with its reflection: 10,548 T
uses the allowance pays for in each preparation: 1,572 to 2,262
for comparison, step 3's walk makes 2,048
if the budget counts a Toffoli as one hard operation instead of 4 T: 6,152 to 8,853 uses


In [65]:
print(f"lambda: {lattice.one_norm_mev:,.0f} MeV")
print(
    f"gamma Delta at least: {strong.rule_of_thumb_mev:.3g} to "
    f"{weak.rule_of_thumb_mev:.3g} MeV"
)
print(
    f"  so with an overlap of 0.5, a gap above {2 * strong.rule_of_thumb_mev:.3g} to "
    f"{2 * weak.rule_of_thumb_mev:.3g} MeV"
)
print(
    f"if a Toffoli counts as one: {strong_counted.rule_of_thumb_mev:.3g} to "
    f"{weak_counted.rule_of_thumb_mev:.3g} MeV"
)

lambda: 21,646 MeV
gamma Delta at least: 9.57 to 13.8 MeV
  so with an overlap of 0.5, a gap above 19.1 to 27.5 MeV
if a Toffoli counts as one: 2.44 to 3.52 MeV


## A lever nobody has priced: normalize the current at $Z$

The slide of this title is in the back-up, and a bonus challenge exercise follows it
there.

What if the current could be block-encoded with $\Lambda_J = Z$, the proton count, in
place of $M$, the site count? The $M$ comes from how the insertion writes the current: a
list of $2M$ terms of size $\tfrac{1}{2}$, a Pauli $Z$ on each proton mode, and a block
encoding divides by the sum of the coefficients' sizes, $2M \times \tfrac{1}{2} = M$,
counted over every site whether a proton sits there or not. But on carbon's states the
current only ever acts on $Z$ protons: it is the sum of $Z$ phases of modulus one, one
per proton, so $\|\rho_{\vec q}|\Psi\rangle\| \le Z$ for every state with $Z$ protons,
and a block encoding normalized at $Z$ would succeed with probability
$P_s = \eta^2/Z^2 \le 1$. The strength band, our estimate, puts $\eta^2$ between $Z/2$
and $Z$. Dividing by $M^2$ in place of $Z^2$ throws away a factor $(M/Z)^2$ in $P_s$,
`gain_at_proton_count`, for nothing. The rounds go as $1/\sqrt{P_s}$, so the rounds and
the preparations fall with the square root of the gain, and the allowance per
preparation, the leftover budget divided by the preparations, rises by about the same
factor. Nothing else in the circuit changes.

Nobody has built that block encoding, and the reason is worth knowing. A linear
combination of unitaries cannot do it: its one-norm is fixed by the terms as written,
not by the state they act on. In first quantization, one position register per proton,
the current is a sum of exactly $Z$ unitaries, a phase on each proton's register, and
the normalization $Z$ comes free; but this calculation is second-quantized, one qubit
per mode, and the walk's block encoding of $H$ with it. A second-quantized version would
need a PREPARE that reads which modes are occupied and puts the index register over
those $Z$ alone, a construction of a different kind from the PREPARE and SELECT the
lecture built, with nothing to price yet. `price_strength_band` with
`current_one_norm=Z` prices the circuit as if it existed, which shows what it would be
worth.

In [66]:
weak_at_z, strong_at_z = price_strength_band(
    nucleus,
    box_side=lattice.box_side,
    register_bits=register.bits,
    current_one_norm=float(nucleus.proton_count),
)
gain = gain_at_proton_count(nucleus, lattice)
print(f"P_s rises by (M / Z)^2 = {gain:.3g}")
print(f"rounds: {strong_at_z.schedule.rounds} to {weak_at_z.schedule.rounds}")
print(
    f"preparations: {strong_at_z.schedule.preparations} to "
    f"{weak_at_z.schedule.preparations}"
)
print(
    f"allowance per preparation: {weak_at_z.allowance_t:.3g} T to "
    f"{strong_at_z.allowance_t:.3g} T"
)

P_s rises by (M / Z)^2 = 114
rounds: 1 to 2
preparations: 3 to 5
allowance per preparation: 1.96e+08 T to 3.26e+08 T


## The error budget (i): two kinds of error

A logical fault is a gate that went wrong despite error correction, with nothing to flag
it. Some faults barely move the answer and some ruin the run; nobody has mapped which
for this circuit, so we count every faulted run as lost. Synthesis error, from the
rotation sections, is a small, known bias in every run, bounded in size. They get
separate budgets: the rotations had `SYNTHESIS_ERROR_BUDGET`, and the machine's logical
faults get `FAULT_BUDGET`, 0.1 expected faults per circuit. Then at most a tenth of the
runs are lost. A histogram tolerates that if the lost runs scatter over the register's
values, which we assume. `logical_error_rate` spreads the fault budget over the $N_T$
hard operations, $\epsilon_l \le 0.1/N_T$ each, leaving out the Clifford gates and the
idle qubits. The code must reach the lower of two rates: the one the priced parts need,
and the one a circuit that fills the budget needs. The last line counts the faults at
the 2028 figure, `LOGICAL_ERROR_2028`, $10^{-8}$ per operation; the request for
information states none for 2033.

In [67]:
faults_priced = expected_faults(priced, logical_error=LOGICAL_ERROR_2028)
faults_full = expected_faults(budget, logical_error=LOGICAL_ERROR_2028)
on_priced = logical_error_rate(priced)
on_budget = logical_error_rate(budget)
print(f"fault budget: {FAULT_BUDGET:g} expected logical faults per circuit")
print(f"logical error rate, on the priced parts: {on_priced:.2e} per operation")
print(f"if a circuit uses the whole budget: {on_budget:.2e} per operation")
print(
    f"at the 2028 figure of {LOGICAL_ERROR_2028:.0e} per operation: "
    f"{faults_priced:.2g} expected faults on the priced parts, "
    f"{faults_full:.2g} if a circuit uses the whole budget"
)

fault budget: 0.1 expected logical faults per circuit
logical error rate, on the priced parts: 4.59e-09 per operation
if a circuit uses the whole budget: 1.00e-10 per operation
at the 2028 figure of 1e-08 per operation: 0.22 expected faults on the priced parts, 10 if a circuit uses the whole budget


## The error budget (ii): what the error-correcting code must deliver

`needed_code_distance` finds the surface code's distance $d$ at the lower of the two
rates, with the physical error ten times below threshold, from the schematic
$\epsilon_l \sim (p/p_{th})^{(d + 1)/2}$ of hour 1 (Fowler et al. 2012, arXiv:1208.0928,
Eq. 11). `physical_qubits_per_logical` takes $2d^2$. The last line counts the physical
qubits of the circuit's largest stage, for the algorithm's qubits alone.

In [68]:
distance = weak.needed_code_distance
peak = weak.qubits.peak
print(f"code distance at the lower of the two rates: {distance}")
print(
    f"physical qubits per logical one, 2 d^2: {physical_qubits_per_logical(distance)}"
)
print(
    f"for the {peak:,}-qubit stage: "
    f"{physical_qubits(peak, distance=distance):.2e} physical qubits"
)

code distance at the lower of the two rates: 19
physical qubits per logical one, 2 d^2: 722
for the 524-qubit stage: 3.78e+05 physical qubits


## The qubit count

How many logical qubits does the machine hold at once? `weak.qubits`, a `QubitCount`,
counts the register by stage. During the walk: the system, the energy register,
PREPARE's ancillas and REFLECT's AND chain. During the amplification: the system, the
current's index and the long reflection's AND chain. The peak is the larger stage.
Carbon's preparation is unpriced, so its ancillas come on top where they outnumber the
chain's ANDs, or where the reflection must cover them, and the peak is a floor. The last
line gives what the 2033 budget's 1,000 logical qubits leave for a preparation. These
are algorithmic qubits, the ones the circuit names, and we read the budget's 1,000 the
same way; the request for information does not say. On the leading architectures a chip
needs more, for routing and for factories, as the back-up slide "From the algorithm's
qubits to a chip: routing and factories" says.

In [69]:
qubits = weak.qubits
print(f"system: {qubits.system:,}")
print("during the walk:")
print(
    f"  energy register {qubits.energy_register}, ancillas {qubits.walk_ancillas}, "
    f"AND chain {qubits.walk_chain}: {qubits.walk_added} added"
)
print(f"  stage total: {qubits.walk_stage:,}")
print("during the amplification:")
print(f"  index: {qubits.insertion_index}")
print(
    f"  AND chain over the {qubits.reflected_qubits:,} system and index qubits, "
    f"one of them the reflection's target: {qubits.amplification_chain:,}"
)
print(f"  {qubits.amplification_added:,} added")
print(f"  stage total: {qubits.amplification_stage:,}")
print(f"peak: {qubits.peak:,}")
print(
    "free for a preparation in the budget: "
    f"{qubits.free_for_preparation} if its ancillas are clean at every reflection, "
    f"{qubits.free_if_reflected_over} if the reflection must cover them"
)

system: 256
during the walk:
  energy register 12, ancillas 43, AND chain 42: 97 added
  stage total: 353
during the amplification:
  index: 7
  AND chain over the 263 system and index qubits, one of them the reflection's target: 261
  268 added
  stage total: 524
peak: 524
free for a preparation in the budget: 737 if its ancillas are clean at every reflection, 238 if the reflection must cover them


## How many runs: one run, one bin

How many circuits does the response take? The response in bin $j$ is the total weight of
the final states whose energies fall in it,
$S_j = \sum_{f \in j} |\langle f|J|\Psi_0\rangle|^2$, and the strength is the total over
all bins, $\eta^2 = \sum_j S_j$, the $\eta^2$ of the section on applying the current. A
successful circuit returns bin $j$ with probability $p_j = S_j/\eta^2$, the bin's share
of the strength, so over $N$ runs the counts are multinomial. `bins_in_window` counts
the bins a response of a given width fills. A peak bin of weight $p$ is pinned to a
fraction $\epsilon$ of itself in $(1 - p)/(p\epsilon^2)$ runs,
`runs_to_pin_bin_to_itself`, or to a fraction $\epsilon$ of the strength in
$p(1 - p)/\epsilon^2$, `runs_to_pin_bin_to_strength`. One histogram is one momentum
transfer and one response function. The lecture's set is a histogram for each of 5
response functions at each of 5 momentum transfers. `runs_for_the_set` counts its
successes, and the circuits it takes when a circuit that misses runs again.

The widths, the bin's weight, the precisions and the set's size are our assumptions,
stated on the slides. The first cell sets them.

In [70]:
NARROWEST_RESPONSE_MEV = 200.0
WIDEST_RESPONSE_MEV = 500.0
PEAK_BIN_WEIGHT = 0.15
PRECISION_OF_THE_BIN = 0.05
PRECISION_OF_THE_STRENGTH = 0.005
RESPONSE_FUNCTIONS = 5
MOMENTUM_TRANSFERS = 5

In [71]:
bin_width = register.bin_width_mev
fewest_bins = bins_in_window(NARROWEST_RESPONSE_MEV, bin_width_mev=bin_width)
most_bins = bins_in_window(WIDEST_RESPONSE_MEV, bin_width_mev=bin_width)
to_itself = runs_to_pin_bin_to_itself(PEAK_BIN_WEIGHT, precision=PRECISION_OF_THE_BIN)
to_strength = runs_to_pin_bin_to_strength(
    PEAK_BIN_WEIGHT, precision=PRECISION_OF_THE_STRENGTH
)
print(
    f"a response {NARROWEST_RESPONSE_MEV:g} to {WIDEST_RESPONSE_MEV:g} MeV wide "
    f"fills {fewest_bins} to {most_bins} bins"
)
print(
    f"a peak bin of weight {PEAK_BIN_WEIGHT:g}, "
    f"to {100 * PRECISION_OF_THE_BIN:g}% of itself: {to_itself:,} runs"
)
print(
    f"the same bin, to {100 * PRECISION_OF_THE_STRENGTH:g}% of the strength: "
    f"{to_strength:,} runs"
)

a response 200 to 500 MeV wide fills 7 to 16 bins
a peak bin of weight 0.15, to 5% of itself: 2,267 runs
the same bin, to 0.5% of the strength: 5,100 runs


In [72]:
fewest = runs_for_the_set(
    response_functions=RESPONSE_FUNCTIONS,
    momentum_transfers=MOMENTUM_TRANSFERS,
    runs_per_histogram=to_itself,
    landing_probability=weak_schedule.landing_probability,
)
most = runs_for_the_set(
    response_functions=RESPONSE_FUNCTIONS,
    momentum_transfers=MOMENTUM_TRANSFERS,
    runs_per_histogram=to_strength,
    landing_probability=weak_schedule.landing_probability,
)
print(
    f"{fewest.histograms} histograms: {fewest.successes:,} to "
    f"{most.successes:,} successful circuits"
)
print(
    "circuits run at the weak end, as one that misses runs again: "
    f"{fewest.circuits:,} to {most.circuits:,}"
)

25 histograms: 56,675 to 127,500 successful circuits
circuits run at the weak end, as one that misses runs again: 56,714 to 127,588


## Wall time: $T$ gates divided by the $T$ rate

How long does it all take on one machine? The rate of $T$ gates sets the machine's
speed, and no roadmap states one. Our assumption is $10^6$ $T$ per second, with two
slower rates beside it, $10^5$ and 7,000, the last that of an early layout (Litinski,
arXiv:1808.02892). `wall_time_seconds` divides one circuit's $T$ by the rate. For the
whole set, `OneCircuit.set_t` counts the $T$ that so many successes cost: a miss is found
at the index, before the energy measurement, so it repeats only the rest of the circuit.

In [73]:
T_PER_SECOND = 10**6
SLOWER_T_PER_SECOND = 10**5
EARLY_LAYOUT_T_PER_SECOND = 7 * 10**3
priced_s = wall_time_seconds(priced, runs=1, t_per_second=T_PER_SECOND)
budget_s = wall_time_seconds(budget, runs=1, t_per_second=T_PER_SECOND)
slower_s = wall_time_seconds(budget, runs=1, t_per_second=SLOWER_T_PER_SECOND)
early_s = wall_time_seconds(budget, runs=1, t_per_second=EARLY_LAYOUT_T_PER_SECOND)
print(f"at {T_PER_SECOND:,} T per second:")
print(f"  one circuit, the priced parts: {priced_s:,.0f} s")
print(f"  one circuit, the whole budget: {budget_s / 60:.0f} minutes")
print(
    f"one circuit, the whole budget, at {SLOWER_T_PER_SECOND:,} T per second: "
    f"{slower_s / 3600:.1f} hours; at {EARLY_LAYOUT_T_PER_SECOND:,}: "
    f"{early_s / 3600:.0f} hours"
)

at 1,000,000 T per second:
  one circuit, the priced parts: 22 s
  one circuit, the whole budget: 17 minutes
one circuit, the whole budget, at 100,000 T per second: 2.8 hours; at 7,000: 40 hours


In [74]:
SECONDS_PER_DAY = 24 * 3600
SECONDS_PER_YEAR = 365.25 * SECONDS_PER_DAY
priced_fewest_s = weak.set_t(priced, successes=fewest.successes) / T_PER_SECOND
priced_most_s = weak.set_t(priced, successes=most.successes) / T_PER_SECOND
budget_fewest_s = weak.set_t(budget, successes=fewest.successes) / T_PER_SECOND
budget_most_s = weak.set_t(budget, successes=most.successes) / T_PER_SECOND
print(
    "the whole set, the priced parts: "
    f"{priced_fewest_s / SECONDS_PER_DAY:,.0f} to "
    f"{priced_most_s / SECONDS_PER_DAY:,.0f} days"
)
print(
    "the whole set, the whole budget: "
    f"{budget_fewest_s / SECONDS_PER_YEAR:.1f} to "
    f"{budget_most_s / SECONDS_PER_YEAR:.1f} years"
)

the whole set, the priced parts: 14 to 32 days
the whole set, the whole budget: 1.8 to 4.0 years


With the fault budget and the $T$ rate in hand, the dial of "Amplitude amplification"
can be read against the machine, as the back-up slide "Step 2's trade: many short
circuits, or one deep one" does. Stop after $m$ rounds, measure the index, and start a
new circuit on a miss. The next cell prints the slide's table for carbon at the weak end
of the strength band, with every preparation spending its whole allowance, and adds the
cheapest stop, $21$ rounds. First comes the depth of one circuit: its $1 + 2m$
preparations, its $T$ gates $N_T$ with the energy measurement, and the logical error
rate that many $T$ gates need, $0.1/N_T$. Then the count: the circuits a success takes
on average and its wall time, where a miss stops at the index and only the success runs
the energy measurement.

A miss starts a new circuit, so faults add up only inside one circuit. A lower logical
error rate allows a deeper circuit, and so more rounds. A higher one may force $m = 0$,
trying and discarding, and then the wall clock pays: hours for a success in place of
minutes. Discarded tries can run in parallel on more machines; the rounds of one circuit
cannot, and only a faster $T$ rate shortens them. The cheapest stop shows here too: at
$21$ rounds the circuit is a quarter shallower than the lecture's, and a success comes
sooner. Held to the whole budget instead, it would leave each preparation an allowance a
third larger, which the cell's last line prints.

In [75]:
trade_theta = math.asin(math.sqrt(weak_schedule.success_probability))
trade_costs = []
for trade_rounds in range(weak_schedule.rounds + 1):
    trade_angle = (1 + 2 * trade_rounds) * trade_theta
    trade_costs.append((1 + 2 * trade_rounds) / math.sin(trade_angle) ** 2)
trade_cheapest = trade_costs.index(min(trade_costs))
print(
    "carbon at the weak end, every preparation at its whole allowance, "
    f"{weak.allowance_t:.3g} T:"
)
print("rounds  preparations  T gates, N_T  needs eps_l  circuits a success  wall time")
for trade_rounds in (0, 2, 10, trade_cheapest, weak_schedule.rounds):
    trade_preparations = 1 + 2 * trade_rounds
    trade_to_the_index_t = (
        trade_preparations * (weak.allowance_t + weak.insertion_t)
        + trade_rounds * weak.round_reflections_t
    )
    trade_circuit_t = trade_to_the_index_t + weak.energy_measurement_t
    trade_chance = math.sin(trade_preparations * trade_theta) ** 2
    trade_seconds = wall_time_seconds(
        trade_to_the_index_t / trade_chance + weak.energy_measurement_t,
        runs=1,
        t_per_second=T_PER_SECOND,
    )
    trade_time = f"{trade_seconds / 60:.0f} minutes"
    if trade_seconds >= 3600:
        trade_time = f"{trade_seconds / 3600:.1f} hours"
    print(
        f"{trade_rounds:6d}  {trade_preparations:12d}  {trade_circuit_t:12.1e}  "
        f"{logical_error_rate(trade_circuit_t):11.1e}  {1 / trade_chance:18,.1f}  "
        f"{trade_time:>10}"
    )
assert math.isclose(trade_circuit_t, budget)
trade_left_over_t = (
    budget
    - weak.energy_measurement_t
    - (1 + 2 * trade_cheapest) * weak.insertion_t
    - trade_cheapest * weak.round_reflections_t
)
print(
    f"held to the budget instead, {trade_cheapest} rounds would leave each "
    f"preparation {trade_left_over_t / (1 + 2 * trade_cheapest):.3g} T, "
    f"against {weak.allowance_t:.3g} T"
)

carbon at the weak end, every preparation at its whole allowance, 1.66e+07 T:
rounds  preparations  T gates, N_T  needs eps_l  circuits a success  wall time
     0             1       3.8e+07      2.6e-09             1,365.3   6.3 hours
     2             5       1.0e+08      9.6e-10                54.9   1.3 hours
    10            21       3.7e+08      2.7e-10                 3.5  20 minutes
    21            43       7.3e+08      1.4e-10                 1.2  14 minutes
    29            59       1.0e+09      1.0e-10                 1.0  17 minutes
held to the budget instead, 21 rounds would leave each preparation 2.27e+07 T, against 1.66e+07 T


## The verdict against the 2033 budget

Where does each case stand? `standing` compares a band's two ends with the 2033 budget,
and `standing_if_one_per_toffoli` does the same if a Toffoli counts as one hard
operation. Each share is of the budget: of the whole circuit where the preparation is
priced, and of all but the preparations where it is not. A priced circuit fits while its
share is at most 1. An unpriced one, like carbon's, fits only in the sense that it is
still open: something is left for its preparations. The lecture's three rows are carbon
on $4^3$, helium on $3^3$, and helium on the next lattice, $4^3$. The budget constrains
one circuit; the runs are a separate axis, and they were the previous sections.

In [76]:
helium_on_4 = LectureCase(
    nucleus=helium.nucleus, box_side=4, sizing_mev=helium.sizing_mev
)
helium_on_4_weak, helium_on_4_strong = price_lecture_case(helium_on_4)
rows = [
    ("carbon, 4^3", weak, strong),
    ("helium, 3^3", helium_weak, helium_strong),
    ("helium, 4^3", helium_on_4_weak, helium_on_4_strong),
]

In [77]:
for name, row_weak, row_strong in rows:
    band = standing(row_weak, row_strong)
    counted = standing_if_one_per_toffoli(row_weak, row_strong)
    print(f"{name}, {row_weak.register.rms_error_mev:.1f} MeV RMS:")
    print(f"  energy measurement: {row_weak.energy_measurement_t:.2e} T")
    print(
        f"  preparations: {row_strong.schedule.preparations} to "
        f"{row_weak.schedule.preparations}"
    )
    print(f"  preparation priced: {band.preparation_priced}")
    print(
        f"  share of the budget: {100 * band.strong_end_share:.1f}% to "
        f"{100 * band.weak_end_share:.1f}%; fits at the strong end: "
        f"{band.strong_end_fits}, at the weak end: {band.weak_end_fits}"
    )
    print(
        f"  if a Toffoli counts as one: {100 * counted.strong_end_share:.1f}% to "
        f"{100 * counted.weak_end_share:.1f}%; fits at the strong end: "
        f"{counted.strong_end_fits}, at the weak end: {counted.weak_end_fits}"
    )
    print(f"  peak qubits: {row_weak.qubits.peak:,}")

carbon, 4^3, 23.5 MeV RMS:
  energy measurement: 2.17e+07 T
  preparations: 41 to 59
  preparation priced: False
  share of the budget: 2.2% to 2.2%; fits at the strong end: True, at the weak end: True
  if a Toffoli counts as one: 0.6% to 0.6%; fits at the strong end: True, at the weak end: True
  peak qubits: 524
helium, 3^3, 19.8 MeV RMS:
  energy measurement: 5.32e+06 T
  preparations: 29 to 43
  preparation priced: True
  share of the budget: 7.9% to 11.5%; fits at the strong end: True, at the weak end: True
  if a Toffoli counts as one: 2.0% to 2.9%; fits at the strong end: True, at the weak end: True
  peak qubits: 286
helium, 4^3, 11.7 MeV RMS:
  energy measurement: 4.36e+07 T
  preparations: 71 to 101
  preparation priced: True
  share of the budget: 279.9% to 396.3%; fits at the strong end: False, at the weak end: False
  if a Toffoli counts as one: 70.0% to 99.1%; fits at the strong end: True, at the weak end: True
  peak qubits: 589


# Closing matter: argon, the roadmap, and reading an estimate

Argon is exercise 4, so the notebook leaves it to the script. Of the closing slides, one
states a number the module gives.

## The RFI and the roadmap

Could the 2028 machine run any of this? The same request for information gives the 2028
machine $10^5$ hard operations per circuit, `HARD_OPERATIONS_2028`, on 150 to 250
logical qubits at $\epsilon_l \le 10^{-8}$. The cell sets carbon's energy measurement
alone against it, and helium's. The 2028 configuration is a pipeline-demonstration
class, with a different, smaller calculation.

In [78]:
print(f"the 2028 budget: {HARD_OPERATIONS_2028:,} hard operations")
print(
    "carbon's energy measurement alone: "
    f"{measurement / HARD_OPERATIONS_2028:.0f} times the budget"
)
print(
    "helium's energy measurement alone: "
    f"{helium_weak.energy_measurement_t / HARD_OPERATIONS_2028:.0f} times the budget"
)

the 2028 budget: 100,000 hard operations
carbon's energy measurement alone: 217 times the budget
helium's energy measurement alone: 53 times the budget


## Exercises

The exercises are on the slides "Exercises (i)" to "Exercises (iii)", and a bonus
challenge is on the back-up slide "Bonus challenge exercise: the lever, for helium". The
script `scripts/price_one_circuit.py` gives the numbers each one asks you to reason
about, and its docstring lists them with the options that run each. Most take an option
or two; exercises 3 and 5 take none, since their numbers are in the script's default
output. `uv run python scripts/price_one_circuit.py --help` lists the options. The
module's docstring names the paper and equation each piece rests on.